# Weak GPU Complexity Benchmark: Mat32 / top_q / N

目标是稳定识别经验复杂度、定位瓶颈、比较算法在弱 GPU 上的拐点，不追求单点最快。

本 notebook 只回答四个问题：

- Q1: `gpu_v1_topq0` 的 **训练时间** `time_train` 随 `N` 如何增长（表中同时可查 `wall_s_total`）。
- Q2: `gpu_v3_topq` 相比 `gpu_v1_topq0` 在不同 `N` 下更快还是更慢，拐点在哪里。
- Q3: **训练路径**内的占比：各阶段相对于 `time_train`（`predict` 仅占 `wall_s_total` 的分量，单列比例）。
- Q4: 随着 `N` 增大，`top_q>0` 相对于 v1 的 **训练耗时**拐点（`time_train` 意义下的 speedup）。

实验拆成三层：

1. **训练耗时** `time_train`（precompute、eigenspace、precond_build、solve）随 `N` 变化作为主图 Fig1；端到端 **`wall_s_total` = `time_train` + `time_predict`**，主要在汇总表里对比
2. 分阶段时间 vs `N`
3. 迭代统计 vs `N`

控制变量固定：

- 数据分布固定（2D + Mat32 + 固定采样分布）
- `eps` 固定
- `cg_tol` 固定
- `N_test` 固定
- 每条曲线内部 `top_q` 固定

输出要求：

- 每次 run 结果会 `print`
- 同时保存 `raw csv`
- 聚合统计保存 `summary csv`
- 图像保存到 `png`


In [45]:
'''
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

# 定义一个方便引用的实验结果保存路径（建议根据项目命名）
# 这会在你的 Google Drive 根目录下创建一个文件夹
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab_Experiments/EFGP_Eigenpro'

if not os.path.exists(DRIVE_OUTPUT_DIR):
    os.makedirs(DRIVE_OUTPUT_DIR)
    print(f"✅ 已在 Drive 中创建目录: {DRIVE_OUTPUT_DIR}")
else:
    print(f"📂 实验结果将同步至: {DRIVE_OUTPUT_DIR}")
'''

'\nfrom google.colab import drive\nimport os\n\n# 挂载 Google Drive\ndrive.mount(\'/content/drive\')\n\n# 定义一个方便引用的实验结果保存路径（建议根据项目命名）\n# 这会在你的 Google Drive 根目录下创建一个文件夹\nDRIVE_OUTPUT_DIR = \'/content/drive/MyDrive/Colab_Experiments/EFGP_Eigenpro\'\n\nif not os.path.exists(DRIVE_OUTPUT_DIR):\n    os.makedirs(DRIVE_OUTPUT_DIR)\n    print(f"✅ 已在 Drive 中创建目录: {DRIVE_OUTPUT_DIR}")\nelse:\n    print(f"📂 实验结果将同步至: {DRIVE_OUTPUT_DIR}")\n'

In [46]:
## For github import
'''
import os
import sys

GITHUB_USER = "Yifiwifi"
REPO_NAME = "EFGP-Eigenpro"
SUB_DIR = "efgp_eigenpro_py"
PROJECT_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(PROJECT_PATH):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
else:
    %cd {PROJECT_PATH}
    !git pull origin main

CODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)
if CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

print("Checking runtime dependencies")
!pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com

requirements_path = os.path.join(CODE_ROOT, "requirements.txt")
if os.path.exists(requirements_path):
    !pip install -r {requirements_path}

sanity_check_path = os.path.join(CODE_ROOT, "gpu/sanity_check")
if os.path.exists(sanity_check_path):
    os.chdir(sanity_check_path)
    print("cwd:", os.getcwd())
else:
    print("sanity_check path not found:", sanity_check_path)

# Refresh runtime library path for some Colab images
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
!ldconfig /usr/local/lib

print("=" * 40)
try:
    import torch
    import cupy as cp
    import cufinufft
    cp.cuda.Stream.null.synchronize()
    print("PyTorch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print("cufinufft import ok")
except Exception as e:
    print("runtime check failed:", e)
print("=" * 40)
'''


'\nimport os\nimport sys\n\nGITHUB_USER = "Yifiwifi"\nREPO_NAME = "EFGP-Eigenpro"\nSUB_DIR = "efgp_eigenpro_py"\nPROJECT_PATH = f"/content/{REPO_NAME}"\n\nif not os.path.exists(PROJECT_PATH):\n    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git\nelse:\n    %cd {PROJECT_PATH}\n    !git pull origin main\n\nCODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)\nif CODE_ROOT not in sys.path:\n    sys.path.append(CODE_ROOT)\n\nprint("Checking runtime dependencies")\n!pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com\n\nrequirements_path = os.path.join(CODE_ROOT, "requirements.txt")\nif os.path.exists(requirements_path):\n    !pip install -r {requirements_path}\n\nsanity_check_path = os.path.join(CODE_ROOT, "gpu/sanity_check")\nif os.path.exists(sanity_check_path):\n    os.chdir(sanity_check_path)\n    print("cwd:", os.getcwd())\nelse:\n    print("sanity_check path not found:", sanity_check_path)\n\n# Refresh runtime library path for some Colab images\nos.env

In [47]:
import time
from datetime import timedelta
start_time = time.time()

In [48]:
import gc
import os
import sys
import time
import json
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py.efgp_solver import EFGPSolver, PrecomputeState
from efgp_eigenpro_py.benchmark import make_dataset, make_test_set, true_func_2d, compute_rmse
from efgp_eigenpro_py.gpu.backends import BackendConfig
from efgp_eigenpro_py.gpu.versions import GPURunConfig, run_v1_pure_efgp, run_v3_full_gpu_eigenspace
from efgp_eigenpro_py.gpu.v3_eigenspace import EigenspaceConfig

try:
    import cupy as cp
except Exception:
    cp = None

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])
print("cupy available:", cp is not None)


cwd: d:\NU\ML\efgp_eigenpro_py\gpu\sanity_check
sys.path[0]: D:\NU\ML
cupy available: True


In [ ]:
# ---- Fixed controls ----
DIM = 2
LENGTHSCALE = 0.1
NU = 1.5
REG_LAMBDA = 0.1

# EPS 作为单个标量默认值；真正 sweep 用 EPS_SWEEP + SWEEP 来声明（避免 EPS=list 导致 float(EPS) 报错）
EPS = 1e-5
EPS_SWEEP = [1e-3, 1e-5, 1e-7]

SOLVE_TOL = 1e-6
GPU_MAXITER = 3000
GPU_NUFFT = "auto"
L2_SCALED = True

N_TEST = 1000
NOISE = 0.02
SEED_TRAIN_BASE = 20260421
SEED_TEST = 1

# Recommended weak-GPU first round
N_LIST = [100_000, 300_000]                    # [100_000, 300_000, 1_000_000, 3_000_000, 10_000_000]
MODE_SPECS = [
    {"mode": "gpu_v1_topq0", "top_q": 0},
    {"mode": "gpu_v3_topq_eigenpro_nystrom", "top_q": 45},
    {"mode": "gpu_v3_topq_eigenpro_nystrom", "top_q": 90},
    {"mode": "gpu_v3_topq_eigenpro_nystrom", "top_q": 180},
    {"mode": "gpu_v3_topq_eigenpro_nystrom", "top_q": 360}
]

REPEATS_POLICY = {
    "small": 1,   # N <= 1e6
    "mid": 1,     # N == 3e6
    "large": 1,    # N >= 1e7
}

WARMUP_N = 100_000
WARMUP_SEED = 7
CLEAR_POOL_PER_N = True

V3_OVERSAMPLE = 16
V3_N_ITER = 3

# ---- GPU precompute comparison: exact cuFINUFFT vs binned C0/C1 ----
# "original" = 直接 GPU exact precompute（gpu_precompute_v1 / cuFINUFFT）；"C0"/"C1"/"C2" = build_binned_efgp_system
PRECOMPUTE_COMPARE_ENABLE = True
PRECOMPUTE_METHODS = ["original", "C1"]  # 默认比较 exact GPU precompute、C0、C1；C2 仅按需加做参考
# precompute-only 只比较各自独立 precompute，不复用别的方法的结果；full_training 分支仍沿用原 notebook 的 rhs 替换口径。
BINNED_QUALITY = "balanced"
BINNED_USE_SPARSE_BINS = False  # 禁用 sparse / dict CPU 风格路径
BINNED_USE_GPU_DENSE_BINS = True  # GPU fused dense bin moments + compaction + batched centers NUFFT
# 纯 binned 对比默认不要混入 exact fallback；若某些 case 想强制不中断再手动改 True。
BINNED_ALLOW_EXACT_NUFFT_FALLBACK = False
# 严禁静默使用 CPU FINUFFT
BINNED_NUFFT_ALLOW_CPU_FALLBACK = False
if bool(BINNED_USE_SPARSE_BINS):
    raise ValueError("此 notebook 当前固定为 GPU-only dense bins；BINNED_USE_SPARSE_BINS 必须为 False.")
if not bool(BINNED_USE_GPU_DENSE_BINS):
    raise ValueError("此 notebook 当前固定为 GPU-only binned precompute；BINNED_USE_GPU_DENSE_BINS 必须为 True.")
if bool(BINNED_NUFFT_ALLOW_CPU_FALLBACK):
    raise ValueError("此 notebook 当前固定为 GPU-only precompute；BINNED_NUFFT_ALLOW_CPU_FALLBACK 必须为 False.")
BINNED_R_USER = None  # None 为自动选 r；也可固定整数增大 r；遇 ratio>1 报错可先调大 r 或启用上一项兜底
PRECOMPUTE_ONLY_CLEAR_POOL_PER_N = True  # precompute-only: 每个 N 结束后执行 _clear_state(clear_pool=True)
PRECOMPUTE_COMPARE_CPU_DIRECT_RHS_REFERENCE = False  # 默认 False：避免 CPU direct sum 把 precompute-only 跑得像卡死
BENCHMARK_AFTER_CASE_GPU_POOL_FLUSH = True  # 主 benchmark：每跑完一个 case 清空 CuPy pool（Colab 减压，略慢）

# ---- Notebook 运行阶段（单一切换）----
# "none": runner 不写 raw_df。
# "precompute_grid": N_LIST×REPEATS×PRECOMPUTE_METHODS 的独立 precompute 计时 CSV（不再按 MODE_SPECS 重复空转）。
# "full_training": warmup + gpu 全流程 + SUMMARY/plots/Q/SLQ。
NB_RUN_STAGE = "full_training"  # none | precompute_grid | full_training

_valid_nb_stages = frozenset({"none", "precompute_grid", "full_training"})
if str(NB_RUN_STAGE) not in _valid_nb_stages:
    raise ValueError(f"NB_RUN_STAGE must be one of {sorted(_valid_nb_stages)}; got {NB_RUN_STAGE!r}")

# ---- Plot: per-mode precompute_method selector ----
# 对每个 mode（见 MODE_SPECS）选择绘图时使用的 precompute_method。
# 可选值："original" / "C0" / "C1"；填 None / [] / "none" 表示不过滤（全画）。
PLOT_PRECOMPUTE_METHODS_BY_MODE = {
    "gpu_v1_topq0": ["original"],
    "gpu_v3_topq": ["original"],
    "gpu_v3_topq_eigenpro_nystrom": ["C1"],
}
PLOT_PRECOMPUTE_METHODS_DEFAULT = None  # 未显式列出的 mode：None 表示全画



# EigenPro Nyström：与 v1_mat32_eps_topq_eigenspace_compare.ipynb 中 EIGENPRO_NYSTROM_* 一致
EIGENPRO_NYSTROM_PRECOND_KIND = "coordinate_nystrom"
EIGENPRO_NYSTROM_SURROGATE_SIZE = 2400  # ~10 * (top_q+1) when top_q=64；设为 None 则 10 * (top_q + 1)
EIGENPRO_NYSTROM_LOWFREQ_RATIO = 0.25
EIGENPRO_NYSTROM_OVERSAMPLE = 10
EIGENPRO_NYSTROM_RITZ_REFINE = False
EIGENPRO_NYSTROM_SEED = 0
EIGENPRO_NYSTROM_BLOCK_ROWS = 8192  # None => auto（见 v3 _auto_block_rows）
EIGENPRO_NYSTROM_RITZ_BLOCK_COLS = 16
EIGENPRO_NYSTROM_LIFT = False  # False：仅 I[:,S]V，不做 K[:,S]T^-1 全网格 lift（与 v1 注释一致）


def make_eigenpro_nystrom_eigenspace_config(top_q: int) -> EigenspaceConfig:
    """主流程 `gpu_v3_topq_eigenpro_nystrom` 的 EigenPro–Nyström 子空间配置（与 v1 一致，不经 SLQ）。"""
    tq = int(top_q)
    if EIGENPRO_NYSTROM_SURROGATE_SIZE is not None:
        s_nys = int(EIGENPRO_NYSTROM_SURROGATE_SIZE)
    else:
        s_nys = 10 * (tq + 1)
    br = EIGENPRO_NYSTROM_BLOCK_ROWS
    return EigenspaceConfig(
        q_max=tq,
        block_size=max(s_nys, tq + 1),
        n_iter=0,
        eig_method="eigenpro_nystrom",
        method_cfg={"precond_kind": str(EIGENPRO_NYSTROM_PRECOND_KIND).lower()},
        surrogate_size=s_nys,
        surrogate_oversample=int(EIGENPRO_NYSTROM_OVERSAMPLE),
        surrogate_lowfreq_ratio=float(EIGENPRO_NYSTROM_LOWFREQ_RATIO),
        surrogate_ritz_refine=bool(EIGENPRO_NYSTROM_RITZ_REFINE),
        surrogate_seed=int(EIGENPRO_NYSTROM_SEED),
        surrogate_block_rows=None if br is None else int(br),
        surrogate_ritz_block_cols=int(EIGENPRO_NYSTROM_RITZ_BLOCK_COLS),
        surrogate_lift=bool(EIGENPRO_NYSTROM_LIFT),
    )

# ---- Outputs (base) ----
OUT_ROOT = Path("outputs")
OUT_PREFIX = "weak_gpu_complexity"

# ---- Sweep controls ----
# 约定：SWEEP 的 key 必须是上面某个配置变量名；value 为 list/tuple 即 sweep。
# 后续想扫别的参数，只需要在这里加一行即可（例如 "REG_LAMBDA": [0.1, 0.01]）。
SWEEP = {
    "EPS": EPS_SWEEP,
}

print("N_LIST:", N_LIST)
print("MODE_SPECS:", MODE_SPECS)
print(
    "EIGENPRO_NYSTROM:",
    f"precond={EIGENPRO_NYSTROM_PRECOND_KIND}, surrogate_size={EIGENPRO_NYSTROM_SURROGATE_SIZE}, "
    f"lowfreq_ratio={EIGENPRO_NYSTROM_LOWFREQ_RATIO}, oversample={EIGENPRO_NYSTROM_OVERSAMPLE}, "
    f"ritz_refine={EIGENPRO_NYSTROM_RITZ_REFINE}, seed={EIGENPRO_NYSTROM_SEED}, "
    f"block_rows={EIGENPRO_NYSTROM_BLOCK_ROWS}, ritz_block_cols={EIGENPRO_NYSTROM_RITZ_BLOCK_COLS}, "
    f"lift={EIGENPRO_NYSTROM_LIFT}",
)



N_LIST: [100000, 300000]
MODE_SPECS: [{'mode': 'gpu_v1_topq0', 'top_q': 0}, {'mode': 'gpu_v3_topq_eigenpro_nystrom', 'top_q': 360}]
EIGENPRO_NYSTROM: precond=coordinate_nystrom, surrogate_size=2400, lowfreq_ratio=0.25, oversample=10, ritz_refine=False, seed=0, block_rows=8192, ritz_block_cols=16, lift=False


In [50]:
import itertools
import re


def _is_sweep_seq(v):
    return isinstance(v, (list, tuple))


def _as_sweep_list(v):
    if v is None:
        return [None]
    if _is_sweep_seq(v):
        lst = list(v)
        return lst if len(lst) > 0 else [None]
    return [v]


def _fmt_tag_value(v):
    if isinstance(v, float):
        s = f"{v:g}"
    else:
        s = str(v)
    s = s.strip().replace(" ", "")
    # keep filenames stable on Windows
    s = re.sub(r"[^0-9A-Za-z._\-]+", "_", s)
    return s


def _build_sweep_points(sweep: dict) -> list[dict]:
    sweep = sweep or {}
    keys = [str(k) for k in sweep.keys()]
    if not keys:
        return [{}]
    lists = [_as_sweep_list(sweep[k]) for k in keys]
    pts = []
    for vs in itertools.product(*lists):
        pts.append({k: v for k, v in zip(keys, vs) if v is not None})
    return pts


def _apply_sweep_params(params: dict) -> None:
    for k, v in (params or {}).items():
        globals()[str(k)] = v


RUN_TAG_BASE = datetime.now().strftime("%Y%m%d_%H%M%S")
SWEEP_POINTS = _build_sweep_points(SWEEP)

SWEEP_RUNS = []
for pt in SWEEP_POINTS:
    parts = []
    for k in sorted(pt.keys()):
        parts.append(f"{k.lower()}{_fmt_tag_value(pt[k])}")
    tag = "_".join(parts)
    run_tag = f"{RUN_TAG_BASE}_{tag}" if tag else RUN_TAG_BASE
    out_dir = OUT_ROOT / f"{OUT_PREFIX}_{run_tag}"
    out_dir.mkdir(parents=True, exist_ok=True)
    SWEEP_RUNS.append(
        {
            "params": dict(pt),
            "RUN_TAG": run_tag,
            "OUT_DIR": out_dir,
            "RAW_CSV": out_dir / "raw_runs.csv",
            "SUMMARY_CSV": out_dir / "summary_by_mode_n.csv",
            "ENV_JSON": out_dir / "env_info.json",
        }
    )

# Set a default active run (useful for interactive inspection)
if SWEEP_RUNS:
    _apply_sweep_params(SWEEP_RUNS[0]["params"])
    RUN_TAG = SWEEP_RUNS[0]["RUN_TAG"]
    OUT_DIR = SWEEP_RUNS[0]["OUT_DIR"]
    RAW_CSV = SWEEP_RUNS[0]["RAW_CSV"]
    SUMMARY_CSV = SWEEP_RUNS[0]["SUMMARY_CSV"]
    ENV_JSON = SWEEP_RUNS[0]["ENV_JSON"]

print("RUN_TAG_BASE:", RUN_TAG_BASE)
print("SWEEP_POINTS:", SWEEP_POINTS)
print("SWEEP_RUNS:", [str(r["OUT_DIR"]) for r in SWEEP_RUNS])


kernel = make_matern(lengthscale=LENGTHSCALE, nu=NU, dim=DIM, variance=1.0)
x_test, y_test = make_test_set(DIM, N_TEST, true_func_2d, seed=SEED_TEST)


def _sync_gpu():
    if cp is not None:
        cp.cuda.Stream.null.synchronize()


def _clear_state(clear_pool=False):
    _sync_gpu()
    gc.collect()
    if cp is not None and clear_pool:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    _sync_gpu()


def _gpu_mem_used_gb():
    if cp is None:
        return np.nan
    try:
        free_b, total_b = cp.cuda.runtime.memGetInfo()
        return float((total_b - free_b) / (1024 ** 3))
    except Exception:
        return np.nan


def _device_name():
    if cp is None:
        return "cpu"
    try:
        return cp.cuda.runtime.getDeviceProperties(0)["name"].decode("utf-8")
    except Exception:
        return "unknown_gpu"


def _pick_repeats(n_train: int) -> int:
    if n_train <= 1_000_000:
        return int(REPEATS_POLICY["small"])
    if n_train >= 10_000_000:
        return int(REPEATS_POLICY["large"])
    return int(REPEATS_POLICY["mid"])


def _collect_env_info():
    info = {
        "timestamp": datetime.now().isoformat(),
        "python": sys.version,
        "platform": sys.platform,
        "device_name": _device_name(),
        "eps": EPS,
        "solve_tol": SOLVE_TOL,
        "gpu_maxiter": GPU_MAXITER,
        "nufft_mode": GPU_NUFFT,
        "l2_scaled": L2_SCALED,
        "dim": DIM,
        "kernel": "Mat32",
        "nu": NU,
        "lengthscale": LENGTHSCALE,
        "reg_lambda": REG_LAMBDA,
        "n_test": N_TEST,
        "n_list": [int(v) for v in N_LIST],
        "mode_specs": MODE_SPECS,
        "eigenpro_nystrom": {
            "precond_kind": EIGENPRO_NYSTROM_PRECOND_KIND,
            "surrogate_size": EIGENPRO_NYSTROM_SURROGATE_SIZE,
            "lowfreq_ratio": EIGENPRO_NYSTROM_LOWFREQ_RATIO,
            "oversample": EIGENPRO_NYSTROM_OVERSAMPLE,
            "ritz_refine": EIGENPRO_NYSTROM_RITZ_REFINE,
            "seed": EIGENPRO_NYSTROM_SEED,
            "block_rows": EIGENPRO_NYSTROM_BLOCK_ROWS,
            "ritz_block_cols": EIGENPRO_NYSTROM_RITZ_BLOCK_COLS,
            "lift": EIGENPRO_NYSTROM_LIFT,
        },
    }
    try:
        import cupy
        info["cupy"] = cupy.__version__
    except Exception:
        info["cupy"] = "unavailable"
    try:
        import cufinufft
        info["cufinufft"] = getattr(cufinufft, "__version__", "unknown")
    except Exception:
        info["cufinufft"] = "unavailable"
    return info


def _extract_common_metrics(diag: dict):
    return {
        "cg_iters": int(diag.get("cg_iters", -1)),
        "cg_relres": float(diag.get("cg_relres", np.nan)),
        "n_matvec": int(diag.get("n_matvec", 0)),
        "t_matvec_total": float(diag.get("t_matvec_total", np.nan)),
        "n_precond": int(diag.get("n_precond", 0)),
        "t_precond_total": float(diag.get("t_precond_total", np.nan)),
        "time_eigenspace": float(diag.get("time_eigenspace", 0.0)),
        "time_precond_build": float(diag.get("time_precond_build", 0.0)),
        "time_solve": float(diag.get("time_solve", np.nan)),
        "time_predict": float(diag.get("time_predict", np.nan)),
        "nufft_stage": str(diag.get("nufft_stage", "")),
        "device_name": str(diag.get("device_name", "")),
        "eig_nystrom_kernel_s": float(diag.get("eig_nystrom_kernel_s", np.nan)),
        "surrogate_tag": str(diag.get("surrogate_tag", "")),
    }


RUN_TAG_BASE: 20260506_150714
SWEEP_POINTS: [{'EPS': 0.001}, {'EPS': 1e-05}, {'EPS': 1e-07}]
SWEEP_RUNS: ['outputs\\weak_gpu_complexity_20260506_150714_eps0.001', 'outputs\\weak_gpu_complexity_20260506_150714_eps1e-05', 'outputs\\weak_gpu_complexity_20260506_150714_eps1e-07']


In [51]:
# Extended benchmark runner + optional binned precompute comparison
import efgp_eigenpro_py.gpu as _gpu_pkg_bm
import efgp_eigenpro_py.gpu.v1_ops as _gpu_v1_ops_bm
import efgp_eigenpro_py.gpu.versions as _gpu_versions_bm
from efgp_eigenpro_py.discretization import basis_weights, choose_grid_params
import importlib
import efgp_eigenpro_py.gpu.binned_efgp_precompute as _binned_pc_mod
from efgp_eigenpro_py.gpu.backends import build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import ensure_gpu_data_context
from efgp_eigenpro_py.gpu.v1_ops import _device_array_to_numpy

_binned_pc_mod = importlib.reload(_binned_pc_mod)
build_binned_efgp_system = _binned_pc_mod.build_binned_efgp_system
direct_fourier_sum_type1 = _binned_pc_mod.direct_fourier_sum_type1

_BENCHMARK_PC_METHOD_ACTIVE = None
_LAST_PC_PATCH_EXTRA = {}


def _install_gpu_rhs_benchmark_patch() -> None:
    global _GPU_PC_ORIGINAL_FN

    # 允许重复运行本单元格来更新 patch。
    # 关键点：先 reload v1_ops 以恢复真正的原版实现，避免把旧 wrapper 当成 original 导致递归/计时污染。
    import importlib as _il
    _il.reload(_gpu_v1_ops_bm)
    _GPU_PC_ORIGINAL_FN = _gpu_v1_ops_bm.gpu_precompute_v1

    def _wrapped(
        backend,
        kernel,
        eps,
        nufft_tol,
        data_ctx,
        op_ctx=None,
        *,
        l2scaled=False,
        force=False,
        chunk_size=None,
    ):
        global _LAST_PC_PATCH_EXTRA
        pcm = (_BENCHMARK_PC_METHOD_ACTIVE or "gpu_exact").strip().lower()

        # 算法计时目标：当 pcm=c0/c1/c2 时，避免先跑一遍 exact gpu_precompute_v1 再覆盖 rhs。
        # 对 original / 非 binned 的情形，仍保持原版路径。
        if pcm not in ("c0", "c1", "c2"):
            t_nu0 = time.perf_counter()
            ctx = _GPU_PC_ORIGINAL_FN(
                backend,
                kernel,
                eps,
                nufft_tol,
                data_ctx,
                op_ctx,
                l2scaled=l2scaled,
                force=force,
                chunk_size=chunk_size,
            )
            t_nufft = float(time.perf_counter() - t_nu0)
            _LAST_PC_PATCH_EXTRA = {
                "t_original_exact_gpu_precompute_v1_s": t_nufft,
                "time_precompute_NUFFT": t_nufft,
                "time_precompute_binned": float(np.nan),
            }
            return ctx

        xp = backend.xp
        ctx = data_ctx
        X = xp.asarray(ctx.x_gpu, dtype=xp.float64)
        y = xp.asarray(ctx.y_gpu, dtype=xp.float64).reshape(-1)
        n = int(X.shape[0])
        dim = int(kernel.dim)

        # 构建 grid / weights / x_center，与 v1_ops.gpu_precompute_v1 一致的输入定义
        x_min = xp.min(X, axis=0)
        x_max = xp.max(X, axis=0)
        L = float(xp.max(x_max - x_min))
        x_center_gpu = (x_min + x_max) / 2.0
        grid = choose_grid_params(kernel, eps, L, l2scaled=l2scaled)
        mtot = int(grid.mtot)
        hm = (mtot - 1) // 2
        if mtot != 2 * hm + 1:
            raise RuntimeError(f"unexpected mtot={mtot}")

        weights_np = np.ascontiguousarray(basis_weights(kernel, grid.xis, grid.h).reshape(-1))
        w_gpu = xp.asarray(weights_np, dtype=xp.float64)
        weights_flat = w_gpu.reshape(-1)
        weights_nd = weights_flat.reshape((mtot,) * dim)

        x_center_np = np.asarray(_device_array_to_numpy(x_center_gpu, np.float64)).reshape(-1)

        # 纯 binned 路径：同时生成 XtXcol（ms_v=2*mtot-1）与 rhs（mtot）并构造 Gf=FFTN(XtXcol)
        t_bin0 = time.perf_counter()
        v_tilde, b_tilde, diag_bin = build_binned_efgp_system(
            X,
            y,
            n,
            dim,
            float(grid.h),
            hm,
            weights_np,
            order=pcm.upper(),
            quality=str(BINNED_QUALITY),
            r=int(BINNED_R_USER) if BINNED_R_USER is not None else None,
            use_sparse_bins=bool(BINNED_USE_SPARSE_BINS),
            use_gpu_dense_bins=bool(BINNED_USE_GPU_DENSE_BINS),
            return_bin_stats=False,
            x_center=x_center_np,
            backend=backend,
            nufft_tol=float(nufft_tol),
            gpu_timing=True,
            input_on_gpu=True,
            assume_normalized=True,
            skip_cpu_validation=True,
            allow_exact_nufft_fallback=bool(BINNED_ALLOW_EXACT_NUFFT_FALLBACK),
            nufft_allow_cpu_fallback=bool(BINNED_NUFFT_ALLOW_CPU_FALLBACK),
        )

        # binned NUFFT 返回扁平化系数；与 gpu_precompute_v1 一致需 reshape 成 (2*mtot-1)^d 再 fftn
        ms_xtx = 2 * int(mtot) - 1
        _exp_modes = int(ms_xtx) ** int(dim)
        _vt_flat = xp.asarray(v_tilde).reshape(-1)
        if int(_vt_flat.size) != _exp_modes:
            raise RuntimeError(
                f"binned v_tilde size {_vt_flat.size} != (2*mtot-1)^d={_exp_modes} "
                f"(mtot={mtot}, dim={dim})"
            )
        xtxcol_gpu = xp.ascontiguousarray(_vt_flat.reshape((ms_xtx,) * int(dim)))
        Gf_gpu = xp.ascontiguousarray(backend.fft.fftn(xtxcol_gpu))

        # 写回 ctx
        exp_sz = int(mtot**dim)
        if cp is None or not isinstance(b_tilde, cp.ndarray):
            raise RuntimeError("GPU-only notebook expects build_binned_efgp_system to return GPU b_tilde.")
        rhs_gpu = b_tilde.reshape(-1).astype(xp.complex128, copy=False)
        if int(rhs_gpu.size) != exp_sz:
            raise RuntimeError(f"b_tilde size {rhs_gpu.size} != mtot**d={exp_sz}")

        ctx.weights_gpu_nd = weights_nd
        ctx.weights_gpu_flat = weights_flat
        ctx.weights_np_flat = np.ascontiguousarray(weights_np.reshape(-1))
        ctx.rhs_gpu = rhs_gpu
        ctx.xtxcol_gpu = xtxcol_gpu
        ctx.gf_gpu = Gf_gpu
        ctx.x_center_gpu = x_center_gpu

        _sync_gpu()
        t_bin_wall = float(time.perf_counter() - t_bin0)

        bd = diag_bin.get("binned_precompute_breakdown_s", None) or {}
        _LAST_PC_PATCH_EXTRA = {
            "t_original_exact_gpu_precompute_v1_s": float(np.nan),
            "time_precompute_NUFFT": float(np.nan),
            "time_precompute_binned": t_bin_wall,
            "precompute_benchmark_note": (
                "binned pcm: build_binned_efgp_system provides XtXcol + rhs, then Gf=FFTN(XtXcol); "
                "no exact gpu_precompute_v1 is executed in this path."
            ),
            "binned_theta_actual": float(diag_bin.get("theta_actual", np.nan)),
            "binned_G": int(diag_bin.get("G", -1)),
            "binned_num_occupied_bins": int(diag_bin.get("num_occupied_bins", -1)),
            "effective_work_ratio": float(diag_bin.get("effective_work_ratio", np.nan)),
            "binned_used_exact_dense_point_nufft": float(bool(diag_bin.get("used_exact_dense_point_nufft", False))),
            "binned_order_bins_final": str(diag_bin.get("order_bins_final", "")),
            "binned_allow_exact_nufft_fallback": float(bool(diag_bin.get("allow_exact_nufft_fallback", False))),
        }
        for _k, _v in bd.items():
            try:
                _LAST_PC_PATCH_EXTRA[str(_k)] = float(_v)
            except (TypeError, ValueError):
                _LAST_PC_PATCH_EXTRA[str(_k)] = float(np.nan)

        ctx.meta.update(
            {
                "mtot": mtot,
                "dim": dim,
                "h": float(grid.h),
                "weight_shape": tuple(int(s) for s in ctx.weights_gpu_nd.shape),
                "gf_shape": tuple(int(s) for s in ctx.gf_gpu.shape),
                "rhs_shape": tuple(int(s) for s in ctx.rhs_gpu.shape),
                "nufft_tol": float(nufft_tol),
                "nufft_stage": f"binned_{pcm}",
                "chunk_size": None,
                "gf_absmax": float(xp.max(xp.abs(ctx.gf_gpu))),
                "debug_finite_checks": bool(ctx.meta.get("debug_finite_checks", False)),
                "rhs_variant": pcm.upper(),
            }
        )

        del X, y, v_tilde, b_tilde, diag_bin, xtxcol_gpu, Gf_gpu, rhs_gpu, x_center_np, weights_np
        return ctx

    _gpu_v1_ops_bm.gpu_precompute_v1 = _wrapped
    _gpu_versions_bm.gpu_precompute_v1 = _wrapped
    if hasattr(_gpu_pkg_bm, "gpu_precompute_v1"):
        _gpu_pkg_bm.gpu_precompute_v1 = _wrapped
    _gpu_v1_ops_bm._benchmark_rhs_patch_installed = True


_install_gpu_rhs_benchmark_patch()


def compare_precompute_methods(x_train, y_train, solver, *, verbose: bool = True) -> pd.DataFrame:
    # precompute-only 比较必须各方法独立完成：original=exact GPU NUFFT；C0/C1/C2=binned GPU。
    methods = [str(m).strip().lower() for m in PRECOMPUTE_METHODS]
    allowed = {"original", "c0", "c1", "c2"}
    bad = [m for m in methods if m not in allowed]
    if bad:
        raise ValueError(f"unsupported PRECOMPUTE_METHODS entries: {bad}")
    if not bool(BINNED_USE_GPU_DENSE_BINS):
        raise ValueError("compare_precompute_methods requires BINNED_USE_GPU_DENSE_BINS=True.")

    x_train_np = np.asarray(x_train, dtype=np.float64)
    y_train_np = np.asarray(y_train, dtype=np.float64).reshape(-1)
    Nloc = int(x_train_np.shape[0])
    L = float(np.max(x_train_np.max(axis=0) - x_train_np.min(axis=0)))
    grid = choose_grid_params(solver.kernel, solver.eps, L, l2scaled=L2_SCALED)
    hm = int(grid.hm)
    h_grid = float(grid.h)
    x_center_np = (x_train_np.min(axis=0) + x_train_np.max(axis=0)) / 2.0
    D_diag = np.asarray(basis_weights(solver.kernel, grid.xis, h_grid).reshape(-1), dtype=np.float64)

    rows = []
    rhs_ref = None
    t_rhs_ref = np.nan
    need_ref = bool(PRECOMPUTE_COMPARE_CPU_DIRECT_RHS_REFERENCE)
    if need_ref:
        modes_rhs = generate_multi_index(hm, DIM)
        t_ref0 = time.perf_counter()
        rhs_ref = direct_fourier_sum_type1(
            x_train_np,
            y_train_np,
            modes_rhs,
            h_grid,
            x_center_np,
            isign=-1,
            max_points_per_chunk=int(PRECOMPUTE_ONLY_DIRSUM_CHUNK),
        )
        t_rhs_ref = float(time.perf_counter() - t_ref0)

    if verbose:
        print(
            "[compare_precompute_methods] methods=",
            methods,
            "| rhs direct-sum ref=",
            "ON" if need_ref else "OFF(default)",
        )

    if "original" in methods:
        _clear_state(clear_pool=False)
        backend_exact = build_gpu_backend_bundle(BackendConfig(nufft=GPU_NUFFT))
        if not getattr(backend_exact, "has_nufft", False) or getattr(backend_exact, "nufft", None) is None:
            raise RuntimeError("original exact GPU precompute requires a GPU NUFFT backend.")
        if hasattr(backend_exact, "allow_cpu_fallback"):
            backend_exact.allow_cpu_fallback = False
        data_ctx = ensure_gpu_data_context(backend_exact, x_train_np, y_train_np)
        t0 = time.perf_counter()
        ctx_exact = _GPU_PC_ORIGINAL_FN(
            backend_exact,
            solver.kernel,
            solver.eps,
            float(getattr(solver, "nufft_tol", 1e-10)),
            data_ctx,
            None,
            l2scaled=L2_SCALED,
            force=True,
            chunk_size=None,
        )
        _sync_gpu()
        t1 = time.perf_counter()
        row_exact = {
            "method": "original",
            "time_s": float(t1 - t0),
            "rhs_rel_err": np.nan,
            "note": "gpu_precompute_v1 exact GPU NUFFT (independent path)",
            "rhs_direct_sum_s": float(t_rhs_ref) if need_ref else np.nan,
            "t_original_exact_gpu_precompute_v1_s": float(t1 - t0),
        }
        if need_ref:
            rhs_exact = cp.asnumpy(ctx_exact.rhs_gpu) if cp is not None else np.asarray(ctx_exact.rhs_gpu)
            row_exact["rhs_rel_err"] = float(
                np.linalg.norm(np.asarray(rhs_exact).reshape(-1) - rhs_ref)
                / (np.linalg.norm(rhs_ref) + 1e-30)
            )
        rows.append(row_exact)
        del backend_exact, data_ctx, ctx_exact

    for tag in ("c0", "c1", "c2"):
        if tag not in methods:
            continue
        _clear_state(clear_pool=False)
        backend_pc = build_gpu_backend_bundle(BackendConfig(nufft=GPU_NUFFT))
        if not getattr(backend_pc, "has_nufft", False) or getattr(backend_pc, "nufft", None) is None:
            raise RuntimeError(f"{tag.upper()} requires a GPU NUFFT backend.")
        if hasattr(backend_pc, "allow_cpu_fallback"):
            backend_pc.allow_cpu_fallback = False
        xp = backend_pc.xp
        Xg = xp.asarray(x_train_np, dtype=xp.float64)
        yg = xp.asarray(y_train_np, dtype=xp.float64)
        order = tag.upper()
        t0 = time.perf_counter()
        _, b_tilde_gpu, diag = build_binned_efgp_system(
            Xg,
            yg,
            Nloc,
            DIM,
            h_grid,
            hm,
            D_diag,
            order=order,
            quality=str(BINNED_QUALITY),
            r=int(BINNED_R_USER) if BINNED_R_USER is not None else None,
            use_sparse_bins=bool(BINNED_USE_SPARSE_BINS),
            use_gpu_dense_bins=bool(BINNED_USE_GPU_DENSE_BINS),
            return_bin_stats=False,
            x_center=x_center_np,
            backend=backend_pc,
            nufft_tol=float(getattr(solver, "nufft_tol", 1e-10)),
            gpu_timing=True,
            input_on_gpu=True,
            assume_normalized=True,
            skip_cpu_validation=True,
            allow_exact_nufft_fallback=bool(BINNED_ALLOW_EXACT_NUFFT_FALLBACK),
            nufft_allow_cpu_fallback=False,
        )
        _sync_gpu()
        t1 = time.perf_counter()
        if cp is None or not isinstance(b_tilde_gpu, cp.ndarray):
            raise RuntimeError("precompute-only compare expects GPU b_tilde from build_binned_efgp_system.")
        row_bm = {
            "method": order,
            "time_s": float(t1 - t0),
            "rhs_rel_err": np.nan,
            "theta_actual": float(diag.get("theta_actual", np.nan)),
            "G": int(diag.get("G", -1)),
            "num_occupied_bins": int(diag.get("num_occupied_bins", -1)),
            "note": str(diag.get("binning_memory_note", "")),
            "rhs_direct_sum_s": float(t_rhs_ref) if need_ref else np.nan,
        }
        if need_ref:
            rhs_apx = cp.asnumpy(b_tilde_gpu).reshape(-1) / np.asarray(D_diag, dtype=np.float64)
            row_bm["rhs_rel_err"] = float(
                np.linalg.norm(rhs_apx - rhs_ref) / (np.linalg.norm(rhs_ref) + 1e-30)
            )
        bd = diag.get("binned_precompute_breakdown_s", None) or {}
        for _k, _v in bd.items():
            try:
                row_bm[str(_k)] = float(_v)
            except (TypeError, ValueError):
                row_bm[str(_k)] = float(np.nan)
        row_bm["effective_work_ratio"] = float(diag.get("effective_work_ratio", np.nan))
        row_bm["binned_used_exact_dense_point_nufft"] = float(bool(diag.get("used_exact_dense_point_nufft", False)))
        row_bm["binned_order_bins_final"] = str(diag.get("order_bins_final", ""))
        row_bm["binned_allow_exact_nufft_fallback"] = float(bool(diag.get("allow_exact_nufft_fallback", False)))
        rows.append(row_bm)
        del backend_pc, Xg, yg, b_tilde_gpu, diag

    df = pd.DataFrame(rows)
    if verbose:
        print("\n=== Precompute comparison (selected methods) ===")
        print(df.to_string(index=False))
    return df


def run_precompute_only_grid() -> pd.DataFrame:
    rows = []
    mode = "precompute_only"
    top_q = 0
    method_names = [str(m).strip().lower() for m in PRECOMPUTE_METHODS]
    for n_train in N_LIST:
        repeats = _pick_repeats(int(n_train))
        print(f"[precompute-only] N={n_train}, repeats={repeats}, methods={method_names}")
        for rep in range(repeats):
            seed_train = int(SEED_TRAIN_BASE + rep)
            x_train = None
            y_train = None
            solver = None
            df_pc = None
            try:
                x_train, y_train = make_dataset(DIM, int(n_train), true_func_2d, noise=NOISE, seed=seed_train)
                solver = EFGPSolver(
                    kernel=kernel,
                    reg_lambda=REG_LAMBDA,
                    eps=EPS,
                    nufft_tol=1e-10,
                    l2scaled=L2_SCALED,
                )
                df_pc = compare_precompute_methods(x_train, y_train, solver, verbose=False)
                for _, rr in df_pc.iterrows():
                    meth = str(rr.get("method", "")).strip().lower()
                    tpc = float(rr.get("time_s", np.nan))
                    rec = {
                        "run_id": f"{RUN_TAG}_{mode}_pcm{meth}_q{top_q}_N{int(n_train)}_rep{rep}",
                        "timestamp": datetime.now().isoformat(),
                        "mode": mode,
                        "top_q": int(top_q),
                        "N": int(n_train),
                        "eps": float(EPS),
                        "repeat_idx": int(rep),
                        "repeat_count": int(repeats),
                        "precompute_method": meth,
                        "time_precompute": tpc,
                        "time_precompute_NUFFT": tpc if meth == "original" else np.nan,
                        "time_precompute_binned": tpc if meth in ("c0", "c1", "c2") else np.nan,
                        "rhs_rel_err": float(rr.get("rhs_rel_err", np.nan)),
                        "theta_actual": float(rr.get("theta_actual", np.nan)),
                        "G": float(rr.get("G", np.nan)),
                        "num_occupied_bins": float(rr.get("num_occupied_bins", np.nan)),
                        "status": "ok",
                        "error": "",
                    }
                    for _ck in rr.index:
                        _sk = str(_ck)
                        _extra = (
                            "rhs_direct_sum_s",
                            "effective_work_ratio",
                            "binned_used_exact_dense_point_nufft",
                            "binned_allow_exact_nufft_fallback",
                            "binned_order_bins_final",
                            "t_original_exact_gpu_precompute_v1_s",
                        )
                        if (
                            not _sk.startswith("t_")
                            and _sk not in _extra
                        ):
                            continue
                        if _sk in rec:
                            continue
                        if _sk == "binned_order_bins_final":
                            rec[_sk] = str(rr[_ck])
                            continue
                        try:
                            rec[_sk] = float(rr[_ck])
                        except (TypeError, ValueError):
                            rec[_sk] = np.nan
                    rows.append(rec)
            except Exception as e:
                for meth in method_names:
                    rows.append(
                        {
                            "run_id": f"{RUN_TAG}_{mode}_pcm{meth}_q{top_q}_N{int(n_train)}_rep{rep}",
                            "timestamp": datetime.now().isoformat(),
                            "mode": mode,
                            "top_q": int(top_q),
                            "N": int(n_train),
                            "eps": float(EPS),
                            "repeat_idx": int(rep),
                            "repeat_count": int(repeats),
                            "precompute_method": meth,
                            "time_precompute": np.nan,
                            "time_precompute_NUFFT": np.nan,
                            "time_precompute_binned": np.nan,
                            "rhs_rel_err": np.nan,
                            "theta_actual": np.nan,
                            "G": np.nan,
                            "num_occupied_bins": np.nan,
                            "status": "error",
                            "error": f"{type(e).__name__}: {e}",
                        }
                    )
            finally:
                del x_train, y_train, solver, df_pc
                _clear_state(clear_pool=False)
                gc.collect()
        if PRECOMPUTE_ONLY_CLEAR_POOL_PER_N:
            _clear_state(clear_pool=True)
    return pd.DataFrame(rows)


def _bench_predict_shell_state(solver: EFGPSolver, x_train: np.ndarray) -> PrecomputeState:
    """
    solver.predict 仅需 grid / weights / x_center；避免在 GPU 训练路径后再跑一遍完整 CPU precompute
    （会分配大 Gf=(2m-1)^d 与 rhs NUFFT，Colab 易 OOM）。
    """
    x = solver._ensure_2d(np.asarray(x_train, dtype=np.float64))
    L = float(np.max(np.max(x, axis=0) - np.min(x, axis=0)))
    grid = choose_grid_params(solver.kernel, solver.eps, L, l2scaled=solver.l2scaled)
    _, x_center = solver._center_and_scale(x, grid.h)
    weights = np.ascontiguousarray(basis_weights(solver.kernel, grid.xis, grid.h).reshape(-1))
    m = int(grid.mtot)
    d = int(solver.kernel.dim)
    nf = int(m**d)
    return PrecomputeState(
        grid=grid,
        weights=weights,
        Gf=np.empty(1, dtype=np.complex128),
        rhs=np.zeros(nf, dtype=np.complex128),
        x_center=np.asarray(x_center, dtype=float),
        toeplitz_ws=None,
        apply_w=None,
    )


def _run_case_once(
    mode: str,
    top_q: int,
    n_train: int,
    seed_train: int,
    warmup_only=False,
    precompute_method: str = "c1",
) -> dict:
    global _BENCHMARK_PC_METHOD_ACTIVE
    pcm_lc = str(precompute_method).strip().lower()
    x_train, y_train = make_dataset(DIM, n_train, true_func_2d, noise=NOISE, seed=seed_train)

    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=EPS,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=None,
        debug_finite_checks=False,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    try:
        _BENCHMARK_PC_METHOD_ACTIVE = pcm_lc
        _sync_gpu()
        mem_before = _gpu_mem_used_gb()
        t0 = time.perf_counter()
        if mode == "gpu_v1_topq0":
            out = run_v1_pure_efgp(solver, x_train, y_train, cfg)
        elif mode == "gpu_v3_topq":
            if int(top_q) <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq")
            eig_cfg = EigenspaceConfig(
                q_max=int(top_q),
                block_size=int(top_q + V3_OVERSAMPLE),
                n_iter=int(V3_N_ITER),
            )
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        elif mode == "gpu_v3_topq_cupy_eigsh":
            if int(top_q) <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq_cupy_eigsh")
            eig_cfg = EigenspaceConfig(
                q_max=int(top_q),
                block_size=int(top_q + V3_OVERSAMPLE),
                n_iter=int(V3_N_ITER),
                method="cupy_eigsh",
                method_cfg={"tol": 1e-6, "oversample": V3_OVERSAMPLE},
            )
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        elif mode == "gpu_v3_topq_rand_subspace_rr":
            if int(top_q) <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq_rand_subspace_rr")
            eig_cfg = EigenspaceConfig(
                q_max=int(top_q),
                block_size=int(top_q + V3_OVERSAMPLE),
                n_iter=int(V3_N_ITER),
                method="rand_subspace_rr",
                method_cfg={"tol": 1e-6, "oversample": V3_OVERSAMPLE, "maxiter": int(V3_N_ITER)},
            )
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        elif mode == "gpu_v3_topq_eigenpro_nystrom":
            if int(top_q) <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq_eigenpro_nystrom")
            eig_cfg = make_eigenpro_nystrom_eigenspace_config(int(top_q))
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        else:
            raise ValueError(f"unsupported mode: {mode}")
        _sync_gpu()
        t1 = time.perf_counter()
        mem_after = _gpu_mem_used_gb()

        if warmup_only:
            try:
                del out
            except NameError:
                pass
            return {"status": "warmup_done"}

        state_cpu = _bench_predict_shell_state(solver, x_train)
        if cp is None:
            beta_cpu = np.asarray(out.beta_gpu)
        else:
            beta_cpu = cp.asnumpy(out.beta_gpu)
        yhat = solver.predict(x_test, beta_cpu, state_cpu)
        rmse = float(compute_rmse(yhat, y_test))
        diag = out.diagnostics
        del beta_cpu, yhat, state_cpu, out
        pcm_tag = pcm_lc.replace(" ", "")
        row = {
            "run_id": f"{RUN_TAG}_{mode}_pcm{pcm_tag}_q{int(top_q)}_N{int(n_train)}_{seed_train}",
            "timestamp": datetime.now().isoformat(),
            "mode": mode,
            "precompute_method": pcm_lc,
            "N": int(n_train),
            "eps": float(EPS),
            "top_q": int(top_q),
            "reg_lambda": float(REG_LAMBDA),
            "cg_tol": float(SOLVE_TOL),
            "wall_s_outer_s": float(t1 - t0),
            "rmse_test": float(rmse),
            "peak_mem_gb": float(np.nanmax([mem_before, mem_after])),
            "status": "ok",
            "error": "",
        }
        row.update(_extract_common_metrics(diag))
        row.update(dict(_LAST_PC_PATCH_EXTRA))
        t_exact_v1 = float(row.get("t_original_exact_gpu_precompute_v1_s", np.nan))
        if not np.isfinite(t_exact_v1):
            t_exact_v1 = float(row.get("time_precompute_NUFFT", np.nan))
        bi_pc = float(row.get("time_precompute_binned", np.nan))
        if pcm_lc in ("c0", "c1", "c2"):
            # 计时口径：binned 作为算法真实 precompute；不再加上额外的 exact v1 开销
            row["time_precompute"] = float(bi_pc)
            row["time_precompute_NUFFT"] = float(np.nan)
        else:
            row["time_precompute"] = float(t_exact_v1)
            row["time_precompute_NUFFT"] = float(t_exact_v1)
        tp_p = row.get("time_precompute")
        tp_s = row.get("time_solve")
        tp_r = row.get("time_predict")
        if tp_p is None or tp_s is None or (isinstance(tp_p, float) and np.isnan(tp_p)) or (isinstance(tp_s, float) and np.isnan(tp_s)):
            row["time_train"] = np.nan
        else:
            row["time_train"] = float(
                float(tp_p)
                + float(row.get("time_eigenspace") or 0.0)
                + float(row.get("time_precond_build") or 0.0)
                + float(tp_s)
            )
        if (
            not isinstance(row["time_train"], (int, float))
            or np.isnan(row["time_train"])
            or tp_r is None
            or (isinstance(tp_r, float) and np.isnan(tp_r))
        ):
            row["wall_s_total"] = np.nan
        else:
            row["wall_s_total"] = float(row["time_train"]) + float(tp_r)
        return row
    finally:
        _BENCHMARK_PC_METHOD_ACTIVE = None
        _LAST_PC_PATCH_EXTRA.clear()
        _clear_state(clear_pool=bool(BENCHMARK_AFTER_CASE_GPU_POOL_FLUSH))


def _run_warmup_for_mode(mode: str, top_q: int, n_ref: int, precompute_method: str):
    pcm_lc = str(precompute_method).strip().lower()
    n_warm = int(min(n_ref, WARMUP_N))
    print(f"warmup start: mode={mode}, top_q={top_q}, pcm={pcm_lc}, N={n_warm}")
    _clear_state(clear_pool=False)
    _ = _run_case_once(
        mode=mode,
        top_q=top_q,
        n_train=n_warm,
        seed_train=WARMUP_SEED,
        warmup_only=True,
        precompute_method=pcm_lc,
    )
    _clear_state(clear_pool=False)
    print(f"warmup done: mode={mode}, top_q={top_q}, pcm={pcm_lc}")


for _run in SWEEP_RUNS:
    _apply_sweep_params(_run.get("params") or {})
    RUN_TAG = _run["RUN_TAG"]
    OUT_DIR = _run["OUT_DIR"]
    RAW_CSV = _run["RAW_CSV"]
    SUMMARY_CSV = _run["SUMMARY_CSV"]
    ENV_JSON = _run["ENV_JSON"]

    print("=" * 80)
    print("SWEEP RUN:", _run.get("params") or {})
    print("RUN_TAG:", RUN_TAG)
    print("OUT_DIR:", OUT_DIR)

    env_info = _collect_env_info()
    ENV_JSON.write_text(json.dumps(env_info, indent=2), encoding="utf-8")
    print("env info saved:", ENV_JSON)

    if PRECOMPUTE_COMPARE_ENABLE:
        _demo_n = int(min(50_000, max(5_000, N_LIST[0] // 10)))
        _seed_pc = int(SEED_TRAIN_BASE)
        print(f"Precompute demo: N={_demo_n}, methods={PRECOMPUTE_METHODS}")
        _x_d, _y_d = make_dataset(DIM, _demo_n, true_func_2d, noise=NOISE, seed=_seed_pc)
        _solver_pc = EFGPSolver(
            kernel=kernel,
            reg_lambda=REG_LAMBDA,
            eps=EPS,
            nufft_tol=1e-10,
            l2scaled=L2_SCALED,
        )
        try:
            precompute_compare_df = compare_precompute_methods(_x_d, _y_d, _solver_pc)
            precompute_compare_csv = OUT_DIR / "precompute_compare_demo.csv"
            precompute_compare_df.to_csv(precompute_compare_csv, index=False)
            print("saved:", precompute_compare_csv)
        except Exception as _e:
            print("precompute compare skipped:", _e)
            traceback.print_exc()


SWEEP RUN: {'EPS': 0.001}
RUN_TAG: 20260506_150714_eps0.001
OUT_DIR: outputs\weak_gpu_complexity_20260506_150714_eps0.001
env info saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\env_info.json
Precompute demo: N=10000, methods=['original', 'C1']
[compare_precompute_methods] methods= ['original', 'c1'] | rhs direct-sum ref= OFF(default)

=== Precompute comparison (selected methods) ===
  method   time_s  rhs_rel_err                                                                                                                                                        note  rhs_direct_sum_s  t_original_exact_gpu_precompute_v1_s  theta_actual      G  num_occupied_bins  t_h2d_xy_s  t_gpu_fused_bin_moments_s  t_compact_occupied_s  t_binned_cufinufft_on_centers_s  t_rhs_D_multiply_s  effective_work_ratio  binned_used_exact_dense_point_nufft binned_order_bins_final  binned_allow_exact_nufft_fallback
original 0.255047          NaN                                                       

In [52]:
raw_df = pd.DataFrame()

if NB_RUN_STAGE in ("full_training", "precompute_grid"):
    for _run in SWEEP_RUNS:
        _apply_sweep_params(_run.get("params") or {})
        RUN_TAG = _run["RUN_TAG"]
        OUT_DIR = _run["OUT_DIR"]
        RAW_CSV = _run["RAW_CSV"]
        SUMMARY_CSV = _run["SUMMARY_CSV"]
        ENV_JSON = _run["ENV_JSON"]

        print("=" * 100)
        print("SWEEP RUN:", _run.get("params") or {})
        print("RUN_TAG:", RUN_TAG)
        print("OUT_DIR:", OUT_DIR)

        if NB_RUN_STAGE == "full_training":
            rows = []

            for spec in MODE_SPECS:
                for pcm in PRECOMPUTE_METHODS:
                    _run_warmup_for_mode(
                        spec["mode"],
                        int(spec["top_q"]),
                        n_ref=int(N_LIST[0]),
                        precompute_method=str(pcm),
                    )

            for n_train in N_LIST:
                print("=" * 80)
                print(f"N={n_train} start")

                if CLEAR_POOL_PER_N:
                    _clear_state(clear_pool=True)

                for spec in MODE_SPECS:
                    mode = str(spec["mode"])
                    top_q = int(spec["top_q"])
                    repeats = _pick_repeats(int(n_train))

                    for pcm in PRECOMPUTE_METHODS:
                        pcm_lc = str(pcm).strip().lower()
                        print("-" * 80)
                        print(
                            f"mode={mode}, top_q={top_q}, precompute_method={pcm_lc}, repeats={repeats}"
                        )

                        for rep in range(repeats):
                            seed_train = int(SEED_TRAIN_BASE + rep)
                            try:
                                _clear_state(clear_pool=False)
                                row = _run_case_once(
                                    mode=mode,
                                    top_q=top_q,
                                    n_train=int(n_train),
                                    seed_train=seed_train,
                                    warmup_only=False,
                                    precompute_method=pcm_lc,
                                )
                                row["repeat_idx"] = int(rep)
                                row["repeat_count"] = int(repeats)
                                rows.append(row)

                                print(
                                    f"ok rep={rep:02d} N={n_train} mode={mode} pcm={pcm_lc} q={top_q} "
                                    f"train={row['time_train']:.4f}s wall_total={row['wall_s_total']:.4f}s "
                                    f"solve={row['time_solve']:.4f}s "
                                    f"iter={row['cg_iters']} rmse={row['rmse_test']:.6e}"
                                )
                            except Exception as e:
                                tb = traceback.format_exc()
                                pcm_tag = pcm_lc.replace(" ", "")
                                err_row = {
                                    "run_id": f"{RUN_TAG}_{mode}_pcm{pcm_tag}_q{top_q}_N{n_train}_rep{rep}",
                                    "timestamp": datetime.now().isoformat(),
                                    "mode": mode,
                                    "precompute_method": pcm_lc,
                                    "N": int(n_train),
                                    "eps": float(EPS),
                                    "top_q": int(top_q),
                                    "reg_lambda": float(REG_LAMBDA),
                                    "cg_tol": float(SOLVE_TOL),
                                    "repeat_idx": int(rep),
                                    "repeat_count": int(repeats),
                                    "wall_s_total": np.nan,
                                    "time_train": np.nan,
                                    "wall_s_outer_s": np.nan,
                                    "time_precompute": np.nan,
                                    "time_precompute_NUFFT": np.nan,
                                    "time_precompute_binned": np.nan,
                                    "time_eigenspace": np.nan,
                                    "time_precond_build": np.nan,
                                    "time_solve": np.nan,
                                    "time_predict": np.nan,
                                    "cg_iters": np.nan,
                                    "cg_relres": np.nan,
                                    "n_matvec": np.nan,
                                    "t_matvec_total": np.nan,
                                    "n_precond": np.nan,
                                    "t_precond_total": np.nan,
                                    "rmse_test": np.nan,
                                    "peak_mem_gb": np.nan,
                                    "nufft_stage": "",
                                    "device_name": _device_name(),
                                    "status": "error",
                                    "error": f"{type(e).__name__}: {e}",
                                    "error_traceback": tb,
                                }
                                rows.append(err_row)
                                print(
                                    f"error rep={rep:02d} N={n_train} mode={mode} pcm={pcm_lc} q={top_q}: {type(e).__name__}: {e}"
                                )

                df_partial = pd.DataFrame(rows)
                df_partial.to_csv(RAW_CSV, index=False)
                print("partial raw csv saved:", RAW_CSV, "rows=", len(df_partial))

            raw_df = pd.DataFrame(rows)
            raw_df.to_csv(RAW_CSV, index=False)
            print("final raw csv:", RAW_CSV)
            print("raw rows:", len(raw_df))

            with pd.option_context("display.max_rows", 200, "display.max_columns", 200):
                display(raw_df.tail(min(len(raw_df), 20)))

            # --- Postprocess: aggregate + plots for this sweep run ---
            try:
                if "precompute_method" not in raw_df.columns:
                    _raw = raw_df.copy()
                    _raw["precompute_method"] = "original"
                else:
                    _raw = raw_df.copy()

                ok_df = _raw[_raw["status"] == "ok"].copy()
                if ok_df.empty:
                    raise RuntimeError("no successful runs, cannot aggregate")

                metrics = [
                    "wall_s_total",
                    "time_train",
                    "time_precompute",
                    "time_precompute_NUFFT",
                    "time_precompute_binned",
                    "t_original_exact_gpu_precompute_v1_s",
                    "t_original_efgp_solver_precompute_s",
                    "t_h2d_xy_s",
                    "t_gpu_fused_bin_moments_s",
                    "t_compact_occupied_s",
                    "t_binned_cufinufft_on_centers_s",
                    "t_rhs_D_multiply_s",
                    "t_exact_dense_point_dual_nufft_s",
                    "t_gpu_to_cpu_copy_s",
                    "effective_work_ratio",
                    "binned_used_exact_dense_point_nufft",
                    "time_eigenspace",
                    "time_precond_build",
                    "time_solve",
                    "time_predict",
                    "cg_iters",
                    "t_matvec_total",
                    "t_precond_total",
                    "rmse_test",
                    "peak_mem_gb",
                ]
                for _mc in metrics:
                    if _mc not in ok_df.columns:
                        ok_df[_mc] = np.nan

                def _quantile(x, q):
                    s = pd.to_numeric(x, errors="coerce")
                    if s.notna().sum() == 0:
                        return np.nan
                    return float(s.quantile(q))

                group_cols = ["mode", "top_q", "precompute_method", "N", "eps"]
                gb = ok_df.groupby(group_cols, dropna=False)

                parts = []
                for m in metrics:
                    part = gb[m].agg(["median", "mean", "std"]).reset_index()
                    part = part.rename(
                        columns={
                            "median": f"{m}_median",
                            "mean": f"{m}_mean",
                            "std": f"{m}_std",
                        }
                    )
                    q10 = gb[m].apply(lambda s: _quantile(s, 0.10)).reset_index(name=f"{m}_p10")
                    q90 = gb[m].apply(lambda s: _quantile(s, 0.90)).reset_index(name=f"{m}_p90")
                    part = part.merge(q10, on=group_cols, how="left")
                    part = part.merge(q90, on=group_cols, how="left")
                    parts.append(part)

                summary_df = parts[0]
                for p in parts[1:]:
                    keep_cols = [c for c in p.columns if c not in group_cols]
                    summary_df = summary_df.merge(p[group_cols + keep_cols], on=group_cols, how="left")

                count_df = _raw.groupby(group_cols, as_index=False).agg(
                    repeat_count=("run_id", "count"),
                    fail_count=("status", lambda x: int((x != "ok").sum())),
                )
                summary_df = summary_df.merge(count_df, on=group_cols, how="left")

                summary_df = (
                    summary_df.sort_values(["mode", "top_q", "precompute_method", "N", "eps"])
                    .reset_index(drop=True)
                    .copy()
                )

                summary_df.to_csv(SUMMARY_CSV, index=False)
                print("summary csv:", SUMMARY_CSV)

                summary_df["logN"] = np.log(summary_df["N"].astype(float))
                summary_df["logT"] = np.log(summary_df["time_train_median"].astype(float))
                summary_df["local_alpha"] = np.nan
                for (mode, top_q, pcm), idx in summary_df.groupby(
                    ["mode", "top_q", "precompute_method"]
                ).groups.items():
                    ids = list(idx)
                    for i in range(len(ids) - 1):
                        i0, i1 = ids[i], ids[i + 1]
                        dlogn = summary_df.loc[i1, "logN"] - summary_df.loc[i0, "logN"]
                        dlogt = summary_df.loc[i1, "logT"] - summary_df.loc[i0, "logT"]
                        summary_df.loc[i1, "local_alpha"] = float(dlogt / dlogn) if dlogn != 0 else np.nan

                summary_df.to_csv(SUMMARY_CSV, index=False)

                from efgp_eigenpro_py.gpu.benchmark_plots import save_complexity_benchmark_plots

                plot_dir = OUT_DIR / "plots"
                try:
                    saved_paths = save_complexity_benchmark_plots(
                        summary_df,
                        plot_dir,
                        dpi=180,
                        show=False,
                        precompute_methods_by_mode=globals().get("PLOT_PRECOMPUTE_METHODS_BY_MODE", None),
                        precompute_methods_default=globals().get("PLOT_PRECOMPUTE_METHODS_DEFAULT", None),
                    )
                except TypeError:
                    # 兼容旧版 save_complexity_benchmark_plots 不支持 selector 参数的情况
                    saved_paths = save_complexity_benchmark_plots(
                        summary_df,
                        plot_dir,
                        dpi=180,
                        show=False,
                    )

                print("all plots saved in:", plot_dir)
                for p in saved_paths:
                    print("saved:", p)
            except Exception as _pp_e:
                print("postprocess skipped:", type(_pp_e).__name__, _pp_e)
                traceback.print_exc()

        else:  # precompute_grid
            print(
                "RUN precompute_grid: independent original/C0/C1 precompute-only CSV + timing table",
            )
            raw_df = run_precompute_only_grid()
            raw_df.to_csv(RAW_CSV, index=False)
            print("precompute-only raw csv:", RAW_CSV, "rows=", len(raw_df))
            if len(raw_df) > 0:
                print("\n=== raw_runs snapshot (same as csv, incl. per-run timing breakdown) ===")
                with pd.option_context(
                    "display.max_rows",
                    500,
                    "display.max_columns",
                    200,
                    "display.width",
                    320,
                ):
                    print(raw_df.to_string(index=False))
            _ok = raw_df[raw_df["status"] == "ok"].copy()
            if not _ok.empty:
                _tbl = (
                    _ok.groupby(["mode", "top_q", "N", "precompute_method"], dropna=False)[
                        "time_precompute"
                    ]
                    .agg(["median", "mean", "std"])
                    .reset_index()
                    .sort_values(["mode", "top_q", "N", "precompute_method"])
                )
                print("\n=== Precompute-only timing table (s) ===")
                print(_tbl.to_string(index=False))

        _run["raw_df"] = raw_df

    # Keep last run exposed as raw_df for interactive inspection
    if SWEEP_RUNS:
        raw_df = SWEEP_RUNS[-1].get("raw_df", pd.DataFrame())
else:
    print("SKIP runner grid (NB_RUN_STAGE=none); raw_df = empty DataFrame")
    raw_df = pd.DataFrame()


SWEEP RUN: {'EPS': 0.001}
RUN_TAG: 20260506_150714_eps0.001
OUT_DIR: outputs\weak_gpu_complexity_20260506_150714_eps0.001
warmup start: mode=gpu_v1_topq0, top_q=0, pcm=original, N=100000
warmup done: mode=gpu_v1_topq0, top_q=0, pcm=original
warmup start: mode=gpu_v1_topq0, top_q=0, pcm=c1, N=100000
warmup done: mode=gpu_v1_topq0, top_q=0, pcm=c1
warmup start: mode=gpu_v3_topq_eigenpro_nystrom, top_q=360, pcm=original, N=100000
warmup done: mode=gpu_v3_topq_eigenpro_nystrom, top_q=360, pcm=original
warmup start: mode=gpu_v3_topq_eigenpro_nystrom, top_q=360, pcm=c1, N=100000
warmup done: mode=gpu_v3_topq_eigenpro_nystrom, top_q=360, pcm=c1
N=100000 start
--------------------------------------------------------------------------------
mode=gpu_v1_topq0, top_q=0, precompute_method=original, repeats=1
ok rep=00 N=100000 mode=gpu_v1_topq0 pcm=original q=0 train=0.5566s wall_total=0.5618s solve=0.5410s iter=331 rmse=2.022368e-03
----------------------------------------------------------------

,run_id,timestamp,mode,precompute_method,N,eps,top_q,reg_lambda,cg_tol,wall_s_outer_s,rmse_test,peak_mem_gb,status,error,cg_iters,cg_relres,n_matvec,t_matvec_total,n_precond,t_precond_total,time_eigenspace,time_precond_build,time_solve,time_predict,nufft_stage,device_name,eig_nystrom_kernel_s,surrogate_tag,t_original_exact_gpu_precompute_v1_s,time_precompute_NUFFT,time_precompute_binned,time_precompute,time_train,wall_s_total,repeat_idx,repeat_count,precompute_benchmark_note,binned_theta_actual,binned_G,binned_num_occupied_bins,effective_work_ratio,binned_used_exact_dense_point_nufft,binned_order_bins_final,binned_allow_exact_nufft_fallback,t_h2d_xy_s,t_gpu_fused_bin_moments_s,t_compact_occupied_s,t_binned_cufinufft_on_centers_s,t_rhs_D_multiply_s
0,20260506_150714_eps0.001_gpu_v1_topq0_pcmorigi...,2026-05-06T15:07:22.046788,gpu_v1_topq0,original,100000,0.001,0,0.1,0.000001,0.563527,0.002022,0.849707,ok,,331,9.833593e-07,332,0.301983,0,NaN,0.000000,0.000000,0.540961,0.005222,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,0.015599,0.015599,NaN,0.015599,0.556560,0.561782,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260506_150714_eps0.001_gpu_v1_topq0_pcmc1_q0...,2026-05-06T15:07:22.843854,gpu_v1_topq0,c1,100000,0.001,0,0.1,0.000001,0.577761,0.002170,0.853613,ok,,339,9.740906e-07,340,0.304662,0,NaN,0.000000,0.000000,0.549617,0.008070,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,NaN,NaN,0.016778,0.016778,0.566395,0.574465,0,1,binned pcm: build_binned_efgp_system provides ...,1.758514,12321.0,12320.0,0.73920,0.0,C1,0.0,0.000181,0.000181,0.001234,0.014271,0.000310
2,20260506_150714_eps0.001_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:24.334967,gpu_v3_topq_eigenpro_nystrom,original,100000,0.001,360,0.1,0.000001,1.277018,0.002024,1.320410,ok,,29,9.468614e-07,30,0.032565,29,0.015882,1.159426,0.000362,0.083421,0.005942,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,1.159279,coordinate_nystrom_s2400_q360_oversample10_low...,0.026074,0.026074,NaN,0.026074,1.269282,1.275224,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260506_150714_eps0.001_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:25.748270,gpu_v3_topq_eigenpro_nystrom,c1,100000,0.001,360,0.1,0.000001,1.172761,0.002171,1.316504,ok,,29,9.997497e-07,30,0.030360,29,0.014206,1.071289,0.000385,0.077151,0.004624,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,1.071135,coordinate_nystrom_s2400_q360_oversample10_low...,NaN,NaN,0.016627,0.016627,1.165452,1.170076,0,1,binned pcm: build_binned_efgp_system provides ...,1.758514,12321.0,12320.0,0.73920,0.0,C1,0.0,0.000173,0.000173,0.001227,0.014223,0.000221
4,20260506_150714_eps0.001_gpu_v1_topq0_pcmorigi...,2026-05-06T15:07:26.855545,gpu_v1_topq0,original,300000,0.001,0,0.1,0.000001,0.752861,0.001238,0.879004,ok,,432,9.965773e-07,433,0.397136,0,NaN,0.000000,0.000000,0.708869,0.008003,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,0.032512,0.032512,NaN,0.032512,0.741381,0.749384,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20260506_150714_eps0.001_gpu_v1_topq0_pcmc1_q0...,2026-05-06T15:07:27.963142,gpu_v1_topq0,c1,300000,0.001,0,0.1,0.000001,0.858757,0.001261,0.890723,ok,,480,8.064610e-07,481,0.452950,0,NaN,0.000000,0.000000,0.813754,0.014122,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,NaN,NaN,0.025253,0.025253,0.839007,0.853128,0,1,binned pcm: build_binned_efgp_system provides ...,1.011370,37249.0,37243.0,0.74486,0.0,C1,0.0,0.000822,0.000823,0.001597,0.021688,0.000242
6,20260506_150714_eps0.001_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:29.537056,gpu_v3_topq_eigenpro_nystrom,original,300000,0.001,360,0.1,0.000001,1.329976,0.001238,1.343848,ok,,37,8.080687e-07,38,0.036370,37,0.016987,1.167498,0.000460,0.091377,0.007884,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,1.167287,coordinate_nystrom_s2400_q360_oversample10_low...,0.060473,0.060473,NaN,0.060473,1.319807,1.327691,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,20260506_150714_eps0.001_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:31.01103

summary csv: outputs\weak_gpu_complexity_20260506_150714_eps0.001\summary_by_mode_n.csv
all plots saved in: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig1_total_time_vs_n_loglog.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig2_stage_vs_n_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig2_stage_vs_n_gpu_v3_topq_eigenpro_nystrom_q360.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig3_cg_iters_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig4_solve_decompose_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig5_stage_share_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig5_stage_share_gpu_v3_topq_eigenpro_nystrom_q360.png
SWEEP RUN: {'EPS': 1e-05}
RUN_TAG: 20260506_150714_eps1e-05
OUT_DIR: outputs\weak_gpu_complexity_2026050

,run_id,timestamp,mode,precompute_method,N,eps,top_q,reg_lambda,cg_tol,wall_s_outer_s,rmse_test,peak_mem_gb,status,error,cg_iters,cg_relres,n_matvec,t_matvec_total,n_precond,t_precond_total,time_eigenspace,time_precond_build,time_solve,time_predict,nufft_stage,device_name,eig_nystrom_kernel_s,surrogate_tag,t_original_exact_gpu_precompute_v1_s,time_precompute_NUFFT,time_precompute_binned,time_precompute,time_train,wall_s_total,repeat_idx,repeat_count,precompute_benchmark_note,binned_theta_actual,binned_G,binned_num_occupied_bins,effective_work_ratio,binned_used_exact_dense_point_nufft,binned_order_bins_final,binned_allow_exact_nufft_fallback,t_h2d_xy_s,t_gpu_fused_bin_moments_s,t_compact_occupied_s,t_binned_cufinufft_on_centers_s,t_rhs_D_multiply_s
0,20260506_150714_eps1e-05_gpu_v1_topq0_pcmorigi...,2026-05-06T15:07:41.236872,gpu_v1_topq0,original,100000,0.00001,0,0.1,0.000001,0.934306,0.002597,0.882910,ok,,408,9.960973e-07,409,0.601619,0,NaN,0.000000,0.000000,0.907130,0.004851,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,0.020092,0.020092,NaN,0.020092,0.927222,0.932073,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260506_150714_eps1e-05_gpu_v1_topq0_pcmc1_q0...,2026-05-06T15:07:42.496591,gpu_v1_topq0,c1,100000,0.00001,0,0.1,0.000001,0.986482,0.002866,0.884863,ok,,422,9.312288e-07,423,0.623877,0,NaN,0.000000,0.000000,0.952488,0.004787,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,NaN,NaN,0.025561,0.025561,0.978049,0.982835,0,1,binned pcm: build_binned_efgp_system provides ...,5.516534,12321.0,12320.0,0.73920,0.0,C1,0.0,0.000180,0.000181,0.001246,0.022731,0.000221
2,20260506_150714_eps1e-05_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:43.991028,gpu_v3_topq_eigenpro_nystrom,original,100000,0.00001,360,0.1,0.000001,1.215566,0.002596,1.337988,ok,,38,8.911764e-07,39,0.058376,38,0.018066,1.070836,0.000349,0.117425,0.004883,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,1.070677,coordinate_nystrom_s2400_q360_oversample10_low...,0.020355,0.020355,NaN,0.020355,1.208964,1.213847,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260506_150714_eps1e-05_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:45.549442,gpu_v3_topq_eigenpro_nystrom,c1,100000,0.00001,360,0.1,0.000001,1.240268,0.002866,1.355566,ok,,38,9.429432e-07,39,0.060032,38,0.018210,1.089101,0.000355,0.118840,0.005579,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,1.088933,coordinate_nystrom_s2400_q360_oversample10_low...,NaN,NaN,0.022760,0.022760,1.231056,1.236636,0,1,binned pcm: build_binned_efgp_system provides ...,5.516534,12321.0,12320.0,0.73920,0.0,C1,0.0,0.000306,0.000306,0.001204,0.019755,0.000252
4,20260506_150714_eps1e-05_gpu_v1_topq0_pcmorigi...,2026-05-06T15:07:47.407155,gpu_v1_topq0,original,300000,0.00001,0,0.1,0.000001,1.405652,0.001812,0.916113,ok,,583,9.082798e-07,584,0.885805,0,NaN,0.000000,0.000000,1.356827,0.010572,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,0.035295,0.035295,NaN,0.035295,1.392122,1.402694,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20260506_150714_eps1e-05_gpu_v1_topq0_pcmc1_q0...,2026-05-06T15:07:49.157504,gpu_v1_topq0,c1,300000,0.00001,0,0.1,0.000001,1.418472,0.001876,0.921973,ok,,583,9.853088e-07,584,0.882905,0,NaN,0.000000,0.000000,1.364366,0.010746,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,NaN,NaN,0.037317,0.037317,1.401683,1.412429,0,1,binned pcm: build_binned_efgp_system provides ...,3.172714,37249.0,37243.0,0.74486,0.0,C1,0.0,0.000816,0.000817,0.001649,0.033193,0.000293
6,20260506_150714_eps1e-05_gpu_v3_topq_eigenpro_...,2026-05-06T15:07:50.790492,gpu_v3_topq_eigenpro_nystrom,original,300000,0.00001,360,0.1,0.000001,1.317642,0.001811,1.361426,ok,,53,7.958772e-07,54,0.081588,53,0.027380,1.097388,0.000416,0.170269,0.008128,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,1.097215,coordinate_nystrom_s2400_q360_oversample10_low...,0.038234,0.038234,NaN,0.038234,1.306306,1.314434,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,20260506_150714_eps1e-05_gpu_v3_topq_eigenpro_...,2026-05-06T

summary csv: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\summary_by_mode_n.csv
all plots saved in: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig1_total_time_vs_n_loglog.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig2_stage_vs_n_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig2_stage_vs_n_gpu_v3_topq_eigenpro_nystrom_q360.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig3_cg_iters_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig4_solve_decompose_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig5_stage_share_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots\fig5_stage_share_gpu_v3_topq_eigenpro_nystrom_q360.png
SWEEP RUN: {'EPS': 1e-07}
RUN_TAG: 20260506_150714_eps1e-07
OUT_DIR: outputs\weak_gpu_complexity_2026050

,run_id,timestamp,mode,precompute_method,N,eps,top_q,reg_lambda,cg_tol,wall_s_outer_s,rmse_test,peak_mem_gb,status,error,cg_iters,cg_relres,n_matvec,t_matvec_total,n_precond,t_precond_total,time_eigenspace,time_precond_build,time_solve,time_predict,nufft_stage,device_name,eig_nystrom_kernel_s,surrogate_tag,t_original_exact_gpu_precompute_v1_s,time_precompute_NUFFT,time_precompute_binned,time_precompute,time_train,wall_s_total,repeat_idx,repeat_count,precompute_benchmark_note,binned_theta_actual,binned_G,binned_num_occupied_bins,effective_work_ratio,binned_used_exact_dense_point_nufft,binned_order_bins_final,binned_allow_exact_nufft_fallback,t_h2d_xy_s,t_gpu_fused_bin_moments_s,t_compact_occupied_s,t_binned_cufinufft_on_centers_s,t_rhs_D_multiply_s
0,20260506_150714_eps1e-07_gpu_v1_topq0_pcmorigi...,2026-05-06T15:09:03.303504,gpu_v1_topq0,original,100000,1.000000e-07,0,0.1,0.000001,19.230850,0.002597,1.185645,ok,,417,9.913978e-07,418,18.392454,0,NaN,0.000000,0.000000,19.112367,0.012693,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,0.103719,0.103719,NaN,0.103719,19.216085,19.228778,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260506_150714_eps1e-07_gpu_v1_topq0_pcmc1_q0...,2026-05-06T15:09:23.040555,gpu_v1_topq0,c1,100000,1.000000e-07,0,0.1,0.000001,19.347206,0.002870,1.234473,ok,,424,9.179680e-07,425,18.401094,0,NaN,0.000000,0.000000,19.118278,0.014777,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,NaN,NaN,0.189430,0.189430,19.307707,19.322485,0,1,binned pcm: build_binned_efgp_system provides ...,17.419479,12321.0,12320.0,0.73920,0.0,C1,0.0,0.000198,0.000199,0.001294,0.166727,0.001189
2,20260506_150714_eps1e-07_gpu_v3_topq_eigenpro_...,2026-05-06T15:09:27.317905,gpu_v3_topq_eigenpro_nystrom,original,100000,1.000000e-07,360,0.1,0.000001,3.873009,0.002598,1.502051,ok,,56,8.738336e-07,57,2.491280,56,0.032169,1.120188,0.000482,2.639049,0.013755,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,1.119985,coordinate_nystrom_s2400_q360_oversample10_low...,0.097856,0.097856,NaN,0.097856,3.857575,3.871330,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260506_150714_eps1e-07_gpu_v3_topq_eigenpro_...,2026-05-06T15:09:31.643077,gpu_v3_topq_eigenpro_nystrom,c1,100000,1.000000e-07,360,0.1,0.000001,3.920673,0.002871,1.621191,ok,,56,9.431267e-07,57,2.489582,56,0.031628,1.089705,0.000356,2.635487,0.012811,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,1.089549,coordinate_nystrom_s2400_q360_oversample10_low...,NaN,NaN,0.157169,0.157169,3.882717,3.895528,0,1,binned pcm: build_binned_efgp_system provides ...,17.419479,12321.0,12320.0,0.73920,0.0,C1,0.0,0.000226,0.000227,0.001387,0.133800,0.001160
4,20260506_150714_eps1e-07_gpu_v1_topq0_pcmorigi...,2026-05-06T15:09:59.614962,gpu_v1_topq0,original,300000,1.000000e-07,0,0.1,0.000001,27.383559,0.001817,1.246191,ok,,604,9.176727e-07,605,26.229423,0,NaN,0.000000,0.000000,27.248294,0.016480,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,0.115624,0.115624,NaN,0.115624,27.363919,27.380399,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20260506_150714_eps1e-07_gpu_v1_topq0_pcmc1_q0...,2026-05-06T15:10:28.837453,gpu_v1_topq0,c1,300000,1.000000e-07,0,0.1,0.000001,28.783378,0.001884,1.287207,ok,,628,8.693376e-07,629,27.468180,0,NaN,0.000000,0.000000,28.539298,0.016106,binned_c1,NVIDIA GeForce RTX 3050 Laptop GPU,NaN,,NaN,NaN,0.200596,0.200596,28.739893,28.755999,0,1,binned pcm: build_binned_efgp_system provides ...,10.018434,37249.0,37243.0,0.74486,0.0,C1,0.0,0.000824,0.000825,0.002121,0.176300,0.001178
6,20260506_150714_eps1e-07_gpu_v3_topq_eigenpro_...,2026-05-06T15:10:33.887261,gpu_v3_topq_eigenpro_nystrom,original,300000,1.000000e-07,360,0.1,0.000001,4.613507,0.001816,1.562598,ok,,73,8.270028e-07,74,3.205592,73,0.042070,1.086227,0.000380,3.394292,0.016382,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,1.086040,coordinate_nystrom_s2400_q360_oversample10_low...,0.113808,0.113808,NaN,0.113808,4.594707,4.611089,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,202

summary csv: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\summary_by_mode_n.csv
all plots saved in: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig1_total_time_vs_n_loglog.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig2_stage_vs_n_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig2_stage_vs_n_gpu_v3_topq_eigenpro_nystrom_q360.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig3_cg_iters_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig4_solve_decompose_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig5_stage_share_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\plots\fig5_stage_share_gpu_v3_topq_eigenpro_nystrom_q360.png


In [53]:
if NB_RUN_STAGE == "full_training":
    if "precompute_method" not in raw_df.columns:
        raw_df = raw_df.copy()
        raw_df["precompute_method"] = "original"

    ok_df = raw_df[raw_df["status"] == "ok"].copy()

    if ok_df.empty:
        raise RuntimeError("no successful runs, cannot aggregate")

    metrics = [
        "wall_s_total",
        "time_train",
        "time_precompute",
        "time_precompute_NUFFT",
        "time_precompute_binned",
        "t_original_exact_gpu_precompute_v1_s",
        "t_original_efgp_solver_precompute_s",
        "t_h2d_xy_s",
        "t_gpu_fused_bin_moments_s",
        "t_compact_occupied_s",
        "t_binned_cufinufft_on_centers_s",
        "t_rhs_D_multiply_s",
        "t_exact_dense_point_dual_nufft_s",
        "t_gpu_to_cpu_copy_s",
        "effective_work_ratio",
        "binned_used_exact_dense_point_nufft",
        "time_eigenspace",
        "time_precond_build",
        "time_solve",
        "time_predict",
        "cg_iters",
        "t_matvec_total",
        "t_precond_total",
        "rmse_test",
        "peak_mem_gb",
    ]

    for _mc in metrics:
        if _mc not in ok_df.columns:
            ok_df[_mc] = np.nan

    def _quantile(x, q):
        s = pd.to_numeric(x, errors="coerce")
        if s.notna().sum() == 0:
            return np.nan
        return float(s.quantile(q))


    group_cols = ["mode", "top_q", "precompute_method", "N", "eps"]
    gb = ok_df.groupby(group_cols, dropna=False)

    parts = []
    for m in metrics:
        part = gb[m].agg(["median", "mean", "std"]).reset_index()
        part = part.rename(columns={
            "median": f"{m}_median",
            "mean": f"{m}_mean",
            "std": f"{m}_std",
        })

        q10 = gb[m].apply(lambda s: _quantile(s, 0.10)).reset_index(name=f"{m}_p10")
        q90 = gb[m].apply(lambda s: _quantile(s, 0.90)).reset_index(name=f"{m}_p90")

        part = part.merge(q10, on=group_cols, how="left")
        part = part.merge(q90, on=group_cols, how="left")
        parts.append(part)

    summary_df = parts[0]
    for p in parts[1:]:
        keep_cols = [c for c in p.columns if c not in group_cols]
        summary_df = summary_df.merge(p[group_cols + keep_cols], on=group_cols, how="left")

    count_df = raw_df.groupby(["mode", "top_q", "precompute_method", "N", "eps"], as_index=False).agg(
        repeat_count=("run_id", "count"),
        fail_count=("status", lambda x: int((x != "ok").sum())),
    )
    summary_df = summary_df.merge(count_df, on=group_cols, how="left")

    summary_df = summary_df.sort_values(["mode", "top_q", "precompute_method", "N", "eps"]).reset_index(drop=True)
    summary_df.to_csv(SUMMARY_CSV, index=False)

    print("summary csv:", SUMMARY_CSV)
    print("summary rows:", len(summary_df))

    # local slopes on median training time (T_train)，端到端总量见 wall_s_total_median
    summary_df["logN"] = np.log(summary_df["N"].astype(float))
    summary_df["logT"] = np.log(summary_df["time_train_median"].astype(float))
    summary_df["local_alpha"] = np.nan

    for (mode, top_q, pcm), idx in summary_df.groupby(["mode", "top_q", "precompute_method"]).groups.items():
        ids = list(idx)
        for i in range(len(ids) - 1):
            i0, i1 = ids[i], ids[i + 1]
            dlogn = summary_df.loc[i1, "logN"] - summary_df.loc[i0, "logN"]
            dlogt = summary_df.loc[i1, "logT"] - summary_df.loc[i0, "logT"]
            summary_df.loc[i1, "local_alpha"] = float(dlogt / dlogn) if dlogn != 0 else np.nan

    summary_df.to_csv(SUMMARY_CSV, index=False)

    main_cols = [
        "mode", "top_q", "precompute_method", "N", "eps",
        "time_train_median", "time_train_p10", "time_train_p90",
        "wall_s_total_median", "wall_s_total_p10", "wall_s_total_p90",
        "time_precompute_median",
        "t_original_exact_gpu_precompute_v1_s_median",
        "time_precompute_NUFFT_median",
        "time_precompute_binned_median",
        "t_h2d_xy_s_median",
        "t_gpu_fused_bin_moments_s_median",
        "t_compact_occupied_s_median",
        "t_binned_cufinufft_on_centers_s_median",
        "t_rhs_D_multiply_s_median",
        "t_exact_dense_point_dual_nufft_s_median",
        "t_gpu_to_cpu_copy_s_median",
        "effective_work_ratio_median",
        "binned_used_exact_dense_point_nufft_median",
        "time_eigenspace_median",
        "time_precond_build_median", "time_solve_median", "time_predict_median",
        "cg_iters_median", "t_matvec_total_median", "t_precond_total_median", "rmse_test_median",
        "peak_mem_gb_median", "repeat_count", "fail_count", "local_alpha",
    ]

    with pd.option_context("display.max_rows", 200, "display.max_columns", 200):
        display(summary_df[main_cols])
else:
    print("SKIP summary aggregation (NB_RUN_STAGE != 'full_training')")
    summary_df = pd.DataFrame()


summary csv: outputs\weak_gpu_complexity_20260506_150714_eps1e-07\summary_by_mode_n.csv
summary rows: 8


,mode,top_q,precompute_method,N,eps,time_train_median,time_train_p10,time_train_p90,wall_s_total_median,wall_s_total_p10,wall_s_total_p90,time_precompute_median,t_original_exact_gpu_precompute_v1_s_median,time_precompute_NUFFT_median,time_precompute_binned_median,t_h2d_xy_s_median,t_gpu_fused_bin_moments_s_median,t_compact_occupied_s_median,t_binned_cufinufft_on_centers_s_median,t_rhs_D_multiply_s_median,t_exact_dense_point_dual_nufft_s_median,t_gpu_to_cpu_copy_s_median,effective_work_ratio_median,binned_used_exact_dense_point_nufft_median,time_eigenspace_median,time_precond_build_median,time_solve_median,time_predict_median,cg_iters_median,t_matvec_total_median,t_precond_total_median,rmse_test_median,peak_mem_gb_median,repeat_count,fail_count,local_alpha
0,gpu_v1_topq0,0,c1,100000,1.000000e-07,19.307707,19.307707,19.307707,19.322485,19.322485,19.322485,0.189430,NaN,NaN,0.189430,0.000198,0.000199,0.001294,0.166727,0.001189,NaN,NaN,0.73920,0.0,0.000000,0.000000,19.118278,0.014777,424.0,18.401094,NaN,0.002870,1.234473,1,0,NaN
1,gpu_v1_topq0,0,c1,300000,1.000000e-07,28.739893,28.739893,28.739893,28.755999,28.755999,28.755999,0.200596,NaN,NaN,0.200596,0.000824,0.000825,0.002121,0.176300,0.001178,NaN,NaN,0.74486,0.0,0.000000,0.000000,28.539298,0.016106,628.0,27.468180,NaN,0.001884,1.287207,1,0,0.362077
2,gpu_v1_topq0,0,original,100000,1.000000e-07,19.216085,19.216085,19.216085,19.228778,19.228778,19.228778,0.103719,0.103719,0.103719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,19.112367,0.012693,417.0,18.392454,NaN,0.002597,1.185645,1,0,NaN
3,gpu_v1_topq0,0,original,300000,1.000000e-07,27.363919,27.363919,27.363919,27.380399,27.380399,27.380399,0.115624,0.115624,0.115624,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,27.248294,0.016480,604.0,26.229423,NaN,0.001817,1.246191,1,0,0.321749
4,gpu_v3_topq_eigenpro_nystrom,360,c1,100000,1.000000e-07,3.882717,3.882717,3.882717,3.895528,3.895528,3.895528,0.157169,NaN,NaN,0.157169,0.000226,0.000227,0.001387,0.133800,0.001160,NaN,NaN,0.73920,0.0,1.089705,0.000356,2.635487,0.012811,56.0,2.489582,0.031628,0.002871,1.621191,1,0,NaN
5,gpu_v3_topq_eigenpro_nystrom,360,c1,300000,1.000000e-07,4.675782,4.675782,4.675782,4.692243,4.692243,4.692243,0.178436,NaN,NaN,0.178436,0.000832,0.000833,0.001328,0.154957,0.001159,NaN,NaN,0.74486,0.0,1.092330,0.000358,3.404658,0.016461,73.0,3.216917,0.040923,0.001882,1.666113,1,0,0.169178
6,gpu_v3_topq_eigenpro_nystrom,360,original,100000,1.000000e-07,3.857575,3.857575,3.857575,3.871330,3.871330,3.871330,0.097856,0.097856,0.097856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.120188,0.000482,2.639049,0.013755,56.0,2.491280,0.032169,0.002598,1.502051,1,0,NaN
7,gpu_v3_topq_eigenpro_nystrom,360,original,300000,1.000000e-07,4.594707,4.594707,4.594707,4.611089,4.611089,4.611089,0.113808,0.113808,0.113808,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.086227,0.000380,3.394292,0.016382,73.0,3.205592,0.042070,0.001816,1.562598,1,0,0.159170


In [54]:
if NB_RUN_STAGE == "full_training":
    from efgp_eigenpro_py.gpu.benchmark_plots import save_complexity_benchmark_plots

    def _normalize_pcm_choice(v):
        if v is None:
            return None
        if isinstance(v, str):
            v = [v]
        out = []
        for s in v:
            ss = str(s).strip()
            if ss == "" or ss.lower() == "none":
                continue
            ss_lc = ss.lower()
            if ss_lc in ("original", "c0", "c1"):
                out.append(ss_lc)
            else:
                raise ValueError(
                    f"PLOT_PRECOMPUTE_METHODS_BY_MODE only supports ['original','C0','C1'] or none; got {ss!r}"
                )
        if not out:
            return None
        seen = set()
        uniq = []
        for x in out:
            if x not in seen:
                uniq.append(x)
                seen.add(x)
        return uniq

    sel_map_raw = globals().get("PLOT_PRECOMPUTE_METHODS_BY_MODE", {}) or {}
    sel_default_raw = globals().get("PLOT_PRECOMPUTE_METHODS_DEFAULT", None)
    sel_map = {str(k): _normalize_pcm_choice(v) for k, v in sel_map_raw.items()}
    sel_default = _normalize_pcm_choice(sel_default_raw)

    def _filter_plot_df(df: pd.DataFrame) -> pd.DataFrame:
        if "precompute_method" not in df.columns:
            return df
        _df = df.copy()
        _df["_pcm_lc"] = _df["precompute_method"].astype(str).str.strip().str.lower()
        kept = []
        for mode, g in _df.groupby("mode", dropna=False):
            m = str(mode)
            choice = sel_map.get(m, sel_default)
            if choice is None:
                kept.append(g)
                continue
            sub = g[g["_pcm_lc"].isin(choice)]
            kept.append(sub if not sub.empty else g)
        out = pd.concat(kept, ignore_index=True)
        return out.drop(columns=["_pcm_lc"], errors="ignore")

    # Sweep-aware plotting: each sweep run gets its own plot group under its OUT_DIR/plots.
    if "SWEEP_RUNS" in globals() and isinstance(SWEEP_RUNS, list) and len(SWEEP_RUNS) > 0:
        for _run in SWEEP_RUNS:
            OUT_DIR = _run.get("OUT_DIR", OUT_DIR)
            plot_dir = OUT_DIR / "plots"

            if "summary_df" in _run and isinstance(_run["summary_df"], pd.DataFrame):
                _sum = _run["summary_df"].copy()
            else:
                _csv = _run.get("SUMMARY_CSV", None)
                if _csv is not None and hasattr(_csv, "exists") and _csv.exists():
                    _sum = pd.read_csv(_csv)
                else:
                    print("SKIP plots for sweep run (missing summary):", _run.get("params"))
                    continue

            plot_summary_df = _filter_plot_df(_sum)
            try:
                saved_paths = save_complexity_benchmark_plots(
                    plot_summary_df,
                    plot_dir,
                    dpi=180,
                    precompute_methods_by_mode=globals().get("PLOT_PRECOMPUTE_METHODS_BY_MODE", None),
                    precompute_methods_default=globals().get("PLOT_PRECOMPUTE_METHODS_DEFAULT", None),
                    show=False,
                )
            except TypeError:
                saved_paths = save_complexity_benchmark_plots(
                    plot_summary_df,
                    plot_dir,
                    dpi=180,
                    show=False,
                )

            print("all plots saved in:", plot_dir, "| sweep params:", _run.get("params"))
            for p in saved_paths:
                print("saved:", p)
    else:
        plot_dir = OUT_DIR / "plots"
        plot_summary_df = _filter_plot_df(summary_df)
        try:
            saved_paths = save_complexity_benchmark_plots(
                plot_summary_df,
                plot_dir,
                dpi=180,
                precompute_methods_by_mode=globals().get("PLOT_PRECOMPUTE_METHODS_BY_MODE", None),
                precompute_methods_default=globals().get("PLOT_PRECOMPUTE_METHODS_DEFAULT", None),
                show=False,
            )
        except TypeError:
            saved_paths = save_complexity_benchmark_plots(
                plot_summary_df,
                plot_dir,
                dpi=180,
                show=False,
            )

        print("all plots saved in:", plot_dir)
        for p in saved_paths:
            print("saved:", p)
else:
    print("SKIP benchmark plots (NB_RUN_STAGE != 'full_training')")


all plots saved in: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots | sweep params: {'EPS': 0.001}
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig1_total_time_vs_n_loglog.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig2_stage_vs_n_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig2_stage_vs_n_gpu_v3_topq_eigenpro_nystrom_q360.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig3_cg_iters_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig4_solve_decompose_vs_n.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig5_stage_share_gpu_v1_topq0_q0.png
saved: outputs\weak_gpu_complexity_20260506_150714_eps0.001\plots\fig5_stage_share_gpu_v3_topq_eigenpro_nystrom_q360.png
all plots saved in: outputs\weak_gpu_complexity_20260506_150714_eps1e-05\plots | sweep params: {'EPS': 1e-05}
saved: outputs\weak_gpu_complexity_20260506_150714_

In [55]:
if NB_RUN_STAGE == "full_training":
    _summary_q_base = summary_df
    if "precompute_method" in summary_df.columns:
        _sm = summary_df["precompute_method"].astype(str).str.lower()
        _summary_q_base = summary_df.loc[_sm == "original"].copy()
        if len(_summary_q_base) < len(summary_df):
            print(
                "Q1-Q4: 仅 precompute_method==original（与 v5 原语义一致）；多方法对比请直接读 summary_df / raw CSV。"
            )

    def _get_curve(df, mode, top_q):
        s = df[(df["mode"] == mode) & (df["top_q"] == top_q)].sort_values("N").copy()
        return s


    v1 = _get_curve(_summary_q_base, "gpu_v1_topq0", 0)
    v3_qs = sorted([int(v) for v in _summary_q_base.loc[_summary_q_base["mode"] == "gpu_v3_topq", "top_q"].dropna().unique()])

    print("=" * 80)
    print("Q1: gpu_v1_topq0 的训练耗时 T_train 随 N 如何增长（表内同时列出 wall_s_total = T_train + T_predict）")
    if len(v1) >= 2:
        print(v1[["N", "time_train_median", "wall_s_total_median", "local_alpha"]])
        print("Q1 observation: local_alpha 趋势见上表，可直接读 log-log 局部斜率。")
    else:
        print("Q1 data insufficient")

    print("=" * 80)
    print("Q2: gpu_v3_topq 相对 gpu_v1_topq0 的训练快慢与拐点（T_train speedup），表内可查 wall_s_total")
    for q in v3_qs:
        v3 = _get_curve(_summary_q_base, "gpu_v3_topq", q)
        m = v1.merge(v3, on=["N", "eps"], suffixes=("_v1", "_v3"))
        if m.empty:
            continue
        m["speedup_v1_over_v3"] = m["time_train_median_v1"] / m["time_train_median_v3"]
        m["v3_faster"] = m["speedup_v1_over_v3"] > 1.0
        print(f"top_q={q}")
        print(m[["N", "time_train_median_v1", "time_train_median_v3", "wall_s_total_median_v1", "wall_s_total_median_v3", "speedup_v1_over_v3", "v3_faster"]])

    print("=" * 80)
    print("Q3: 训练阶段各占 T_train 的比例（predict 仅占 wall_s_total 的一部分，见 ratio_time_predict_median_vs_wall_total）")
    for (mode, top_q), g in _summary_q_base.groupby(["mode", "top_q"]):
        g = g.sort_values("N").copy()
        denom_train = g["time_train_median"].replace(0, np.nan)
        for c in ["time_precompute_median", "time_eigenspace_median", "time_precond_build_median", "time_solve_median"]:
            g[f"ratio_{c}_of_train"] = g[c] / denom_train
        g["ratio_time_predict_median_vs_wall_total"] = g["time_predict_median"] / g["wall_s_total_median"].replace(0, np.nan)
        print(f"mode={mode}, top_q={int(top_q)}")
        print(g[[
            "N",
            "ratio_time_precompute_median_of_train",
            "ratio_time_eigenspace_median_of_train",
            "ratio_time_precond_build_median_of_train",
            "ratio_time_solve_median_of_train",
            "ratio_time_predict_median_vs_wall_total",
        ]])

    print("=" * 80)
    print("Q4: top_q>0 优势扩大还是缩小（比较 T_train）")
    for q in v3_qs:
        v3 = _get_curve(_summary_q_base, "gpu_v3_topq", q)
        m = v1.merge(v3, on=["N", "eps"], suffixes=("_v1", "_v3"))
        if m.empty:
            continue
        m["speedup_v1_over_v3"] = m["time_train_median_v1"] / m["time_train_median_v3"]
        print(f"top_q={q}, speedup trend")
        print(m[["N", "speedup_v1_over_v3"]])

    print("=" * 80)
    print("Artifacts")
    print("raw csv:", RAW_CSV)
    print("summary csv:", SUMMARY_CSV)
    print("plots dir:", plot_dir)
    print("env info:", ENV_JSON)
else:
    print("SKIP Q-table printout (NB_RUN_STAGE != 'full_training')")


Q1-Q4: 仅 precompute_method==original（与 v5 原语义一致）；多方法对比请直接读 summary_df / raw CSV。
Q1: gpu_v1_topq0 的训练耗时 T_train 随 N 如何增长（表内同时列出 wall_s_total = T_train + T_predict）
        N  time_train_median  wall_s_total_median  local_alpha
2  100000          19.216085            19.228778          NaN
3  300000          27.363919            27.380399     0.321749
Q1 observation: local_alpha 趋势见上表，可直接读 log-log 局部斜率。
Q2: gpu_v3_topq 相对 gpu_v1_topq0 的训练快慢与拐点（T_train speedup），表内可查 wall_s_total
Q3: 训练阶段各占 T_train 的比例（predict 仅占 wall_s_total 的一部分，见 ratio_time_predict_median_vs_wall_total）
mode=gpu_v1_topq0, top_q=0
        N  ratio_time_precompute_median_of_train  \
2  100000                               0.005397   
3  300000                               0.004225   

   ratio_time_eigenspace_median_of_train  \
2                                    0.0   
3                                    0.0   

   ratio_time_precond_build_median_of_train  ratio_time_solve_median_of_train  \
2                        

如果你在本地仓库直接运行（例如 `D:/NU/ML`），可以跳过 `## For github import` 单元，直接从导入单元开始执行。

In [56]:
# ---- Selective SLQ spectral diagnostics on cross-selected (mode_spec, N) ----
import re
from dataclasses import asdict, is_dataclass

import importlib
import efgp_eigenpro_py.gpu.slq_diagnostics as slq_diag
import efgp_eigenpro_py.gpu.slq_pcg_spectrum as slq_pcg_spectrum

slq_diag = importlib.reload(slq_diag)
slq_pcg_spectrum = importlib.reload(slq_pcg_spectrum)
build_slq_matvec_for_benchmark_mode = slq_pcg_spectrum.build_slq_matvec_for_benchmark_mode
SLQLanczosConfig = slq_diag.SLQLanczosConfig
run_slq_lanczos_diagnostic = slq_diag.run_slq_lanczos_diagnostic
summarize_slq_diagnostics = slq_diag.summarize_slq_diagnostics
save_slq_plots = slq_diag.save_slq_plots

# 1) Select experiment types (leave empty to use all MODE_SPECS)
# SLQ 不包含 gpu_v3_topq_eigenpro_nystrom（见下自动过滤）
SLQ_SELECTED_MODE_SPECS = [
    {"mode": "gpu_v1_topq0", "top_q": 0},
    {"mode": "gpu_v3_topq", "top_q": 64}
]

# 2) Select N values (leave empty to use all N_LIST)
SLQ_SELECTED_N_LIST = [100_000]

# 3) SLQ controls
SLQ_CFG = SLQLanczosConfig(
    nv=32,
    k_max=400,
    hermitian_type="complex",
    seed=0,
    breakdown_abs_tol=1e-14,
    breakdown_rel_tol=1e-12,
    reorth_mode="none",
    reorth_window=8,
    reorth_passes=2,
    sync_timing=True,
)
SLQ_SUMMARY_MODE = "spd"  # "spd" or "hermitian"
SLQ_PREFIX_STEPS = list(range(20, SLQ_CFG.k_max + 1, 20))
SLQ_SEED_BASE = 99173

SLQ_DIR = OUT_DIR / "slq_diagnostics"
SLQ_DIR.mkdir(parents=True, exist_ok=True)


def _sanitize_tag(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z_\-]+", "_", str(s)).strip("_")


def _spec_key(spec: dict) -> tuple[str, int, str]:
    mode = str(spec.get("mode", ""))
    top_q = int(spec.get("top_q", -1))
    return (mode, top_q, "")


def _to_jsonable(obj):
    if is_dataclass(obj):
        return _to_jsonable(asdict(obj))
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj


def _pick_mode_specs(all_specs: list[dict], selected_specs: list[dict]) -> list[dict]:
    if len(selected_specs) == 0:
        return list(all_specs)

    # Keep explicit user selection even when (mode, top_q) is not in MODE_SPECS
    # (e.g. SLQ-only probes like q=180/360).
    out = []
    seen = set()

    def _push(spec: dict):
        k = _spec_key(spec)
        if k in seen:
            return
        out.append(spec)
        seen.add(k)

    all_by_key = {_spec_key(s): s for s in all_specs}
    for s in selected_specs:
        k = _spec_key(s)
        if k in all_by_key:
            _push(all_by_key[k])
        else:
            _push(dict(s))
    return out


def _pick_n_list(all_n: list[int], selected_n: list[int]) -> list[int]:
    if len(selected_n) == 0:
        return [int(v) for v in all_n]
    selected_set = {int(v) for v in selected_n}
    return [int(v) for v in all_n if int(v) in selected_set]


def _q_lookup(q_map: dict, q: float) -> float:
    if not isinstance(q_map, dict):
        return float("nan")
    if q in q_map:
        return float(q_map[q])
    key = str(q)
    if key in q_map:
        return float(q_map[key])
    for k, v in q_map.items():
        try:
            if abs(float(k) - float(q)) < 1e-15:
                return float(v)
        except Exception:
            pass
    return float("nan")


def _normalize_summary_schema(summary: dict) -> tuple[dict, dict, dict]:
    """
    Support both old schema {config, run_diagnostics, prefix, final}
    and new schema {raw, derived, views}.
    """
    if isinstance(summary, dict) and all(k in summary for k in ("raw", "derived", "views")):
        return summary.get("raw", {}), summary.get("derived", {}), summary.get("views", {})

    if not isinstance(summary, dict):
        return {}, {}, {}

    final = summary.get("final", {}) if isinstance(summary.get("final", {}), dict) else {}
    prefix = summary.get("prefix", []) if isinstance(summary.get("prefix", []), list) else []
    q_map = final.get("quantiles", {}) if isinstance(final.get("quantiles", {}), dict) else {}

    raw_part = {
        "config": summary.get("config", {}),
        "run_diagnostics": summary.get("run_diagnostics", {}),
        "x_grid": np.asarray(final.get("grid", {}).get("x", []), dtype=float),
        "x_scale": str(final.get("grid", {}).get("x_scale", "linear")),
        "cdf": np.asarray(final.get("grid", {}).get("cdf", []), dtype=float),
        "density": final.get("grid", {}).get("density", {}),
        "extremal_ritz_cloud": final.get("extremal_ritz_cloud", {}),
    }

    q01 = _q_lookup(q_map, 0.01)
    q05 = _q_lookup(q_map, 0.05)
    q95 = _q_lookup(q_map, 0.95)
    q99 = _q_lookup(q_map, 0.99)
    q999 = _q_lookup(q_map, 0.999)
    bw_ratio = float((q95 - q05) / max(abs(_q_lookup(q_map, 0.5)), 1e-30)) if np.isfinite(q95) and np.isfinite(q05) else float("nan")
    tail_ratio = float((q999 - q99) / max(q99 - q01, 1e-30)) if np.isfinite(q999) and np.isfinite(q99) and np.isfinite(q01) else float("nan")
    spike_ratio = float(final.get("lambda_hat_max", np.nan) / max(q999, 1e-30)) if np.isfinite(q999) else float("nan")

    derived_part = {
        "prefix": prefix,
        "final_quantiles": q_map,
        "lambda_hat_min": float(final.get("lambda_hat_min", np.nan)),
        "lambda_hat_max": float(final.get("lambda_hat_max", np.nan)),
        "final_kappa_eff": final.get("kappa_eff", {}),
        "final_spread_eff": final.get("spread_eff", {}),
        "final_near_zero_mass": final.get("near_zero_mass", {}),
        "final_tail_ratios": {
            "bulk_width_ratio": bw_ratio,
            "tail_ratio": tail_ratio,
            "spike_ratio": spike_ratio,
        },
    }

    views_part = {
        "headline": {
            "health": "legacy",
            "dominant_issue": "",
        }
    }
    return raw_part, derived_part, views_part


def _save_slq_plots(case_dir: Path, raw_part: dict, derived_part: dict):
    plot_dir = case_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    # Plotting is centralized in slq_diagnostics.py; notebook only prepares payload.
    _summary_pack = {
        "raw": raw_part if isinstance(raw_part, dict) else {},
        "derived": derived_part if isinstance(derived_part, dict) else {},
        "views": {},
    }
    save_slq_plots(_summary_pack, str(plot_dir), dpi=160)
    return


def _build_slq_matvec_and_size(
    x_train: np.ndarray, y_train: np.ndarray, spec: dict
) -> tuple:
    # SLQ 使用的 matvec 与对应 benchmark 模式一致: v1 为 A, v3 为 P(A v) 与 PCG 相同
    mode = str(spec.get("mode", ""))
    top_q = int(spec.get("top_q", 0))
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=EPS,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=None,
        debug_finite_checks=False,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )
    return build_slq_matvec_for_benchmark_mode(
        mode,
        solver,
        x_train,
        y_train,
        cfg,
        top_q=top_q,
        combo_cfg=None,
        v3_oversample=V3_OVERSAMPLE,
        v3_n_iter=V3_N_ITER,
        dim=int(DIM),
    )

if NB_RUN_STAGE == "full_training":
    picked_specs = _pick_mode_specs(MODE_SPECS, SLQ_SELECTED_MODE_SPECS)
    picked_specs = [
        s
        for s in picked_specs
        if str(s.get("mode", "")) != "gpu_v3_topq_eigenpro_nystrom"
    ]
    picked_n = _pick_n_list(N_LIST, SLQ_SELECTED_N_LIST)

    print("SLQ selected mode specs:")
    for s in picked_specs:
        print("  ", s)
    print("SLQ selected N list:", picked_n)

    if len(picked_specs) == 0:
        raise ValueError(
            "No SLQ mode left: SLQ ignores gpu_v3_topq_eigenpro_nystrom; add other modes to "
            "SLQ_SELECTED_MODE_SPECS or MODE_SPECS."
        )
    if len(picked_n) == 0:
        raise ValueError("No N selected for SLQ. Check SLQ_SELECTED_N_LIST.")

    slq_rows = []
    case_idx = 0
    for n_train in picked_n:
        for spec in picked_specs:
            case_idx += 1
            mode = str(spec.get("mode", ""))
            top_q = int(spec.get("top_q", 0))
            case_seed = int(SLQ_SEED_BASE + case_idx)
            print("=" * 100)
            print(f"[SLQ] start N={n_train}, mode={mode}, top_q={top_q}, seed={case_seed}")

            x_train, y_train = make_dataset(DIM, int(n_train), true_func_2d, noise=NOISE, seed=case_seed)
            backend, matvec, size, slq_op_meta = _build_slq_matvec_and_size(x_train, y_train, spec)

            _spec_mode = (
                str(slq_op_meta.get("slq_spectrum", "")) if isinstance(slq_op_meta, dict) else ""
            )
            _spectrum_mode = (
                "hermitian" if _spec_mode == "M_inv_A" else SLQ_SUMMARY_MODE
            )

            t0 = time.perf_counter()
            slq_res = run_slq_lanczos_diagnostic(
                backend=backend,
                matvec=matvec,
                size=size,
                cfg=SLQ_CFG,
            )
            summary = summarize_slq_diagnostics(
                slq_res,
                prefix_steps=SLQ_PREFIX_STEPS,
                spectrum_mode=_spectrum_mode,
            )
            if isinstance(summary, dict) and isinstance(summary.get("raw"), dict) and isinstance(
                slq_op_meta, dict
            ):
                summary["raw"]["slq_operator_meta"] = _to_jsonable(slq_op_meta)
            t1 = time.perf_counter()

            raw_part, derived_part, views_part = _normalize_summary_schema(summary)

            lam_min = float(derived_part.get("lambda_hat_min", np.nan))
            lam_max = float(derived_part.get("lambda_hat_max", np.nan))
            q_map = derived_part.get("final_quantiles", {}) if isinstance(derived_part, dict) else {}
            q01 = _q_lookup(q_map, 0.01)
            q99 = _q_lookup(q_map, 0.99)

            mode_tag = _sanitize_tag(mode)
            case_tag = f"N{int(n_train)}_{mode_tag}_q{int(top_q)}_seed{case_seed}"
            case_dir = SLQ_DIR / case_tag
            case_dir.mkdir(parents=True, exist_ok=True)

            np.savez_compressed(
                case_dir / "lanczos_coeffs.npz",
                alpha=slq_res.alpha,
                beta=slq_res.beta,
                active_steps=slq_res.active_steps,
            )

            (case_dir / "slq_summary.json").write_text(
                json.dumps(_to_jsonable(summary), indent=2),
                encoding="utf-8",
            )

            x_grid = np.asarray(raw_part.get("x_grid", []), dtype=float)
            cdf_grid = np.asarray(raw_part.get("cdf", []), dtype=float)
            density_dict = raw_part.get("density", {}) if isinstance(raw_part, dict) else {}
            grid_df = pd.DataFrame({"x": x_grid, "cdf": cdf_grid})
            if isinstance(density_dict, dict):
                for fac, den in density_dict.items():
                    col = f"density_sigmafac_{fac}"
                    grid_df[col] = np.asarray(den, dtype=float)
            grid_df.to_csv(case_dir / "slq_grid.csv", index=False)

            _save_slq_plots(case_dir, raw_part, derived_part)

            row = {
                "case_tag": case_tag,
                "N": int(n_train),
                "mode": mode,
                "top_q": int(top_q),
                "slq_operator": (
                    str(slq_op_meta.get("slq_spectrum", ""))
                    if isinstance(slq_op_meta, dict)
                    else ""
                ),
                "seed": int(case_seed),
                "size": int(size),
                "wall_s": float(t1 - t0),
                "lambda_hat_min": lam_min,
                "lambda_hat_max": lam_max,
                "q01": q01,
                "q99": q99,
                "m_final": int(derived_part.get("prefix", [])[-1].get("m", SLQ_CFG.k_max)) if (isinstance(derived_part, dict) and len(derived_part.get("prefix", [])) > 0) else int(SLQ_CFG.k_max),
                "health": str(views_part.get("headline", {}).get("health", "")) if isinstance(views_part, dict) else "",
                "dominant_issue": str(views_part.get("headline", {}).get("dominant_issue", "")) if isinstance(views_part, dict) else "",
                "out_dir": str(case_dir),
            }
            slq_rows.append(row)

            print(f"[SLQ] done case={case_tag}")
            print(f"      lambda_hat_min={lam_min:.6e}, lambda_hat_max={lam_max:.6e}, q01={q01:.6e}, q99={q99:.6e}")
            print(f"      health={row['health']}, dominant_issue={row['dominant_issue']}")
            print(f"      saved: {case_dir}")
            if not (np.isfinite(lam_min) and np.isfinite(lam_max) and np.isfinite(q01) and np.isfinite(q99)):
                print("      warning: NaN detected in compact metrics, check slq_summary.json and schema normalization.")

    slq_df = pd.DataFrame(slq_rows)
    slq_csv = SLQ_DIR / "slq_cases_summary.csv"
    slq_df.to_csv(slq_csv, index=False)
    print("=" * 100)
    print("SLQ cross-selected diagnostics finished.")
    print("summary csv:", slq_csv)
    print(slq_df)
else:
    print("SKIP SLQ diagnostics (NB_RUN_STAGE != 'full_training')")


SLQ selected mode specs:
   {'mode': 'gpu_v1_topq0', 'top_q': 0}
   {'mode': 'gpu_v3_topq', 'top_q': 64}
SLQ selected N list: [100000]
[SLQ] start N=100000, mode=gpu_v1_topq0, top_q=0, seed=99174


KeyboardInterrupt: 

In [ ]:
import os
import time
from datetime import timedelta
from google.colab import files, runtime

# --- 1. 结束计时 ---
end_time = time.time()
try:
    # 确保你在最开头运行了 start_time = time.time()
    elapsed_total = end_time - start_time
    time_str = str(timedelta(seconds=int(elapsed_total)))
except NameError:
    time_str = "未知（未检测到 start_time）"

# --- 2. 配置路径 ---
local_folder = 'outputs' # 你的本地输出文件夹
zip_name = 'experiment_results_20260506.zip'
# 使用开头挂载好的 DRIVE_OUTPUT_DIR
target_drive_path = os.path.join(DRIVE_OUTPUT_DIR, zip_name)

# --- 3. 核心执行逻辑 ---
if os.path.exists(local_folder):
    print(f"📦 正在打包 {local_folder} ...")
    !zip -r -q {zip_name} {local_folder}
    
    # A. 先通过 Linux 命令强制同步到已挂载的 Drive (这是阻塞操作，最安全)
    print(f"💾 正在同步压缩包到 Google Drive...")
    !cp {zip_name} "{target_drive_path}"
    
    # B. 检测 Drive 文件是否存在（非手动检测）
    if os.path.exists(target_drive_path):
        print("✅ 检测到 Drive 同步成功！数据已安全。")
        
        # C. 触发浏览器下载备份
        print("📥 启动浏览器备份下载...")
        files.download(zip_name)
        
        # 打印本次实验信息
        print("-" * 30)
        print(f"📊 实验状态: 成功完成并备份")
        print(f"⏱️ 累计运行耗时: {time_str}")
        print("-" * 30)
        
        # 留出 20 秒给浏览器响应下载请求（哪怕失败，Drive 里也已经有了）
        print("等待 20 秒后将自动断开 GPU 以节省点数...")
        time.sleep(20)
        
        # D. 主动断开
        runtime.unassign()
    else:
        print("❌ 警告：同步到 Drive 失败，为了防止数据丢失，已取消自动断开。请手动检查！")
else:
    print(f"❌ 错误：未找到文件夹 {local_folder}，请检查路径。")

ModuleNotFoundError: No module named 'google.colab'